In [1]:
import pandas as pd
import numpy as np

In [2]:
from pathlib import Path

DATA_DIR = Path("data/processed")

In [4]:
daily_data = pd.read_csv(
    DATA_DIR / "canada_integrated_daily.csv",
    parse_dates=[
        "Date",
        "Bank_Availability_Date",
        "Macro_Reference_Date",
        "Macro_Availability_Date"
    ]
)

daily_data = daily_data.sort_values(
    ["Ticker", "Date"]
).reset_index(drop=True)

print("Shape:", daily_data.shape)
print("Tickers:", daily_data["Ticker"].nunique())
print("Date Range:", daily_data["Date"].min(), "to", daily_data["Date"].max())

daily_data.head()

Shape: (63574, 32)
Tickers: 10
Date Range: 2000-01-04 00:00:00 to 2026-11-06 00:00:00


,Date,Ticker,Open,High,Low,Close,Adj Close,Volume,Dividends,Stock Splits,...,Participation_Rate,Unemployment_Rate,BCPI_Total,BCPI_NonEnergy,BCPI_Energy,BCPI_Metals,BCPI_Forestry,BCPI_Agriculture,BCPI_Fish,Macro_Availability_Date
0,2000-01-04,ABX.TO,25.150000,25.65,24.950001,25.450001,17.804482,926500,0.0,0.0,...,65.5,7.2,270.58,217.41,515.33,265.25,289.41,136.81,883.95,2000-01-01
1,2000-01-05,ABX.TO,25.400000,26.00,25.350000,25.650000,17.944401,1452200,0.0,0.0,...,65.5,7.2,270.58,217.41,515.33,265.25,289.41,136.81,883.95,2000-01-01
2,2000-01-06,ABX.TO,25.799999,26.00,25.500000,25.799999,18.049345,600200,0.0,0.0,...,65.5,7.2,270.58,217.41,515.33,265.25,289.41,136.81,883.95,2000-01-01
3,2000-01-07,ABX.TO,25.850000,26.00,25.100000,25.250000,17.664574,1020800,0.0,0.0,...,65.5,7.2,270.58,217.41,515.33,265.25,289.41,136.81,883.95,2000-01-01
4,2000-01-10,ABX.TO,25.350000,25.35,25.150000,25.250000,17.664574,1281100,0.0,0.0,...,65.5,7.2,270.58,217.41,515.33,265.25,289.41,136.81,883.95,2000-01-01


# Feature Engineering

In [4]:
daily_data["Future_Return_5D"] = (
    daily_data.groupby("Ticker")["Adj Close"]
    .transform(lambda x: (x.shift(-5) / x - 1) * 100)
)

In [5]:
print("Target Statistics:")
print(daily_data["Future_Return_5D"].describe())

print("\nMissing Target Values:")
print(daily_data["Future_Return_5D"].isna().sum())

print("\nSample:")
display(
    daily_data[
        ["Date", "Ticker", "Adj Close", "Future_Return_5D"]
    ].head(10)
)

Target Statistics:
count    63524.000000
mean         0.362051
std          4.238297
min        -47.013487
25%         -1.666637
50%          0.373576
75%          2.314759
max         43.282065
Name: Future_Return_5D, dtype: float64

Missing Target Values:
50

Sample:


,Date,Ticker,Adj Close,Future_Return_5D
0,2000-01-04,ABX.TO,17.804482,0.589415
1,2000-01-05,ABX.TO,17.944401,-1.364461
2,2000-01-06,ABX.TO,18.049345,-3.100806
3,2000-01-07,ABX.TO,17.664574,-2.376280
4,2000-01-10,ABX.TO,17.664574,-1.188113
5,2000-01-11,ABX.TO,17.909424,-0.976560
6,2000-01-12,ABX.TO,17.699556,-1.185829
7,2000-01-13,ABX.TO,17.489670,-0.999986
8,2000-01-14,ABX.TO,17.244814,0.000000
9,2000-01-17,ABX.TO,17.454699,-3.807638


In [6]:
daily_data["Daily_Return"] = (
    daily_data.groupby("Ticker")["Adj Close"]
    .pct_change() * 100
)

In [7]:
print(daily_data["Daily_Return"].describe())

print(
    "\nMissing Values:",
    daily_data["Daily_Return"].isna().sum()
)

display(
    daily_data[
        ["Date", "Ticker", "Adj Close", "Daily_Return"]
    ].head(10)
)

count    63564.000000
mean         0.073344
std          1.945237
min        -29.180444
25%         -0.768343
50%          0.074152
75%          0.902516
max         32.462666
Name: Daily_Return, dtype: float64

Missing Values: 10


,Date,Ticker,Adj Close,Daily_Return
0,2000-01-04,ABX.TO,17.804482,NaN
1,2000-01-05,ABX.TO,17.944401,0.785866
2,2000-01-06,ABX.TO,18.049345,0.584830
3,2000-01-07,ABX.TO,17.664574,-2.131775
4,2000-01-10,ABX.TO,17.664574,0.000000
5,2000-01-11,ABX.TO,17.909424,1.386109
6,2000-01-12,ABX.TO,17.699556,-1.171827
7,2000-01-13,ABX.TO,17.489670,-1.185829
8,2000-01-14,ABX.TO,17.244814,-1.400003
9,2000-01-17,ABX.TO,17.454699,1.217088


In [8]:
lags = [1, 5, 10, 20]

for lag in lags:
    daily_data[f"Return_Lag_{lag}"] = (
        daily_data.groupby("Ticker")["Daily_Return"]
        .shift(lag)
    )

In [9]:
lag_columns = [
    f"Return_Lag_{lag}" for lag in lags
]

print("Missing Values:")
print(daily_data[lag_columns].isna().sum())

display(
    daily_data[
        ["Date", "Ticker", "Daily_Return"] + lag_columns
    ].head(25)
)

Missing Values:
Return_Lag_1      20
Return_Lag_5      60
Return_Lag_10    110
Return_Lag_20    210
dtype: int64


,Date,Ticker,Daily_Return,Return_Lag_1,Return_Lag_5,Return_Lag_10,Return_Lag_20
0,2000-01-04,ABX.TO,NaN,NaN,NaN,NaN,NaN
1,2000-01-05,ABX.TO,0.785866,NaN,NaN,NaN,NaN
2,2000-01-06,ABX.TO,0.584830,0.785866,NaN,NaN,NaN
3,2000-01-07,ABX.TO,-2.131775,0.584830,NaN,NaN,NaN
4,2000-01-10,ABX.TO,0.000000,-2.131775,NaN,NaN,NaN
5,2000-01-11,ABX.TO,1.386109,0.000000,NaN,NaN,NaN
6,2000-01-12,ABX.TO,-1.171827,1.386109,0.785866,NaN,NaN
7,2000-01-13,ABX.TO,-1.185829,-1.171827,0.584830,NaN,NaN
8,2000-01-14,ABX.TO,-1.400003,-1.185829,-2.131775,NaN,NaN
9,2000-01-17,ABX.TO,1.217088,-1.400003,0.000000,NaN,NaN


In [10]:
windows = [5, 10, 20]

for window in windows:
    daily_data[f"Volatility_{window}D"] = (
        daily_data.groupby("Ticker")["Daily_Return"]
        .transform(
            lambda x: x.rolling(window=window).std()
        )
    )

In [11]:
volatility_columns = [
    f"Volatility_{window}D" for window in windows
]

print("Missing Values:")
print(daily_data[volatility_columns].isna().sum())

display(
    daily_data[
        ["Date", "Ticker", "Daily_Return"] +
        volatility_columns
    ].head(30)
)

Missing Values:
Volatility_5D      50
Volatility_10D    100
Volatility_20D    200
dtype: int64


,Date,Ticker,Daily_Return,Volatility_5D,Volatility_10D,Volatility_20D
0,2000-01-04,ABX.TO,NaN,NaN,NaN,NaN
1,2000-01-05,ABX.TO,0.785866,NaN,NaN,NaN
2,2000-01-06,ABX.TO,0.584830,NaN,NaN,NaN
3,2000-01-07,ABX.TO,-2.131775,NaN,NaN,NaN
4,2000-01-10,ABX.TO,0.000000,NaN,NaN,NaN
5,2000-01-11,ABX.TO,1.386109,1.355289,NaN,NaN
6,2000-01-12,ABX.TO,-1.171827,1.398729,NaN,NaN
7,2000-01-13,ABX.TO,-1.185829,1.352683,NaN,NaN
8,2000-01-14,ABX.TO,-1.400003,1.176417,NaN,NaN
9,2000-01-17,ABX.TO,1.217088,1.403159,NaN,NaN


In [12]:
windows = [5, 10, 20, 50]

for window in windows:
    daily_data[f"MA_{window}"] = (
        daily_data.groupby("Ticker")["Adj Close"]
        .transform(
            lambda x: x.rolling(window=window).mean()
        )
    )

In [13]:
ma_columns = [
    f"MA_{window}" for window in windows
]

print("Missing Values:")
print(daily_data[ma_columns].isna().sum())

display(
    daily_data[
        ["Date", "Ticker", "Adj Close"] + ma_columns
    ].head(60)
)

Missing Values:
MA_5      40
MA_10     90
MA_20    190
MA_50    490
dtype: int64


,Date,Ticker,Adj Close,MA_5,MA_10,MA_20,MA_50
0,2000-01-04,ABX.TO,17.804482,NaN,NaN,NaN,NaN
1,2000-01-05,ABX.TO,17.944401,NaN,NaN,NaN,NaN
2,2000-01-06,ABX.TO,18.049345,NaN,NaN,NaN,NaN
3,2000-01-07,ABX.TO,17.664574,NaN,NaN,NaN,NaN
4,2000-01-10,ABX.TO,17.664574,17.825475,NaN,NaN,NaN
5,2000-01-11,ABX.TO,17.909424,17.846463,NaN,NaN,NaN
6,2000-01-12,ABX.TO,17.699556,17.797495,NaN,NaN,NaN
7,2000-01-13,ABX.TO,17.489670,17.685559,NaN,NaN,NaN
8,2000-01-14,ABX.TO,17.244814,17.601608,NaN,NaN,NaN
9,2000-01-17,ABX.TO,17.454699,17.559632,17.692554,NaN,NaN


In [14]:
for window in [5, 10, 20, 50]:
    daily_data[f"MA_Ratio_{window}"] = (
        (daily_data["Adj Close"] / daily_data[f"MA_{window}"] - 1)
        * 100
    )

In [15]:
ratio_columns = [
    f"MA_Ratio_{window}" for window in [5, 10, 20, 50]
]

print("Missing Values:")
print(daily_data[ratio_columns].isna().sum())

display(
    daily_data[
        ["Date", "Ticker", "Adj Close"] +
        ma_columns + ratio_columns
    ].tail(20)
)

Missing Values:
MA_Ratio_5      40
MA_Ratio_10     90
MA_Ratio_20    190
MA_Ratio_50    490
dtype: int64


,Date,Ticker,Adj Close,MA_5,MA_10,MA_20,MA_50,MA_Ratio_5,MA_Ratio_10,MA_Ratio_20,MA_Ratio_50
63554,2026-10-09,TD.TO,164.190002,163.914783,165.498634,167.289614,166.797690,0.167904,-0.790721,-1.852842,-1.563384
63555,2026-10-13,TD.TO,167.830002,164.267673,165.438722,167.326196,166.822055,2.168612,1.445417,0.301092,0.604205
63556,2026-10-14,TD.TO,167.570007,164.401724,165.429279,167.313529,166.835858,1.927159,1.294045,0.153292,0.440043
63557,2026-10-15,TD.TO,170.020004,166.144003,165.813800,167.432301,166.901441,2.332917,2.536703,1.545522,1.868506
63558,2026-10-16,TD.TO,170.259995,167.974002,166.216362,167.485610,166.937065,1.360920,2.432752,1.656491,1.990528
63559,2026-10-19,TD.TO,169.610001,169.058002,166.486392,167.451303,166.953334,0.326515,1.876194,1.289149,1.591263
63560,2026-10-20,TD.TO,171.710007,169.834003,167.050838,167.391901,167.024910,1.104610,2.789073,2.579639,2.805029
63561,2026-10-21,TD.TO,175.110001,171.342001,167.871863,167.571518,167.173821,2.199110,4.311704,4.498666,4.747262
63562,2026-10-22,TD.TO,175.139999,172.366000,169.255002,167.962179,167.317970,1.609366,3.477001,4.273474,4.674949
63563,2026-10-23,TD.TO,174.470001,173.208002,170.591002,168.310898,167.391516,0.728603,2.273859,3.659361,4.228700


In [16]:
momentum_periods = [5, 10, 20]

for period in momentum_periods:
    daily_data[f"Momentum_{period}D"] = (
        daily_data.groupby("Ticker")["Adj Close"]
        .transform(
            lambda x: (x / x.shift(period) - 1) * 100
        )
    )

In [17]:
momentum_columns = [
    f"Momentum_{period}D" for period in momentum_periods
]

print("Missing Values:")
print(daily_data[momentum_columns].isna().sum())

display(
    daily_data[
        ["Date", "Ticker", "Adj Close"] + momentum_columns
    ].tail(20)
)

Missing Values:
Momentum_5D      50
Momentum_10D    100
Momentum_20D    200
dtype: int64


,Date,Ticker,Adj Close,Momentum_5D,Momentum_10D,Momentum_20D
63554,2026-10-09,TD.TO,164.190002,-1.629442,-3.139369,-1.424154
63555,2026-10-13,TD.TO,167.830002,1.062502,-0.355711,0.437843
63556,2026-10-14,TD.TO,167.570007,0.401593,-0.056325,-0.150948
63557,2026-10-15,TD.TO,170.020004,5.400453,2.313958,1.416942
63558,2026-10-16,TD.TO,170.259995,5.679346,2.421653,0.630161
63559,2026-10-19,TD.TO,169.610001,3.301052,1.617822,-0.402911
63560,2026-10-20,TD.TO,171.710007,2.311866,3.398932,-0.687139
63561,2026-10-21,TD.TO,175.110001,4.499608,4.919271,2.094447
63562,2026-10-22,TD.TO,175.139999,3.011407,8.574490,4.669436
63563,2026-10-23,TD.TO,174.470001,2.472693,8.292471,4.163918


In [18]:
# Price changes
delta = daily_data.groupby("Ticker")["Adj Close"].diff()

# Gains and losses
gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)

# Average gain and loss (Wilder's smoothing)
avg_gain = gain.groupby(daily_data["Ticker"]).transform(
    lambda x: x.ewm(
        alpha=1/14,
        adjust=False,
        min_periods=14
    ).mean()
)

avg_loss = loss.groupby(daily_data["Ticker"]).transform(
    lambda x: x.ewm(
        alpha=1/14,
        adjust=False,
        min_periods=14
    ).mean()
)

# RSI
rs = avg_gain / avg_loss

daily_data["RSI_14"] = 100 - (100 / (1 + rs))

# Handle zero-loss cases
daily_data.loc[
    (avg_loss == 0) & (avg_gain > 0), "RSI_14"
] = 100

daily_data.loc[
    (avg_loss == 0) & (avg_gain == 0), "RSI_14"
] = 50

In [19]:
print(daily_data["RSI_14"].describe())

print("\nMissing Values:", daily_data["RSI_14"].isna().sum())

display(
    daily_data[
        ["Date", "Ticker", "Adj Close", "RSI_14"]
    ].tail(20)
)

count    63434.000000
mean        53.383992
std         11.979229
min          7.385830
25%         44.983766
50%         53.561471
75%         61.739231
max         90.940117
Name: RSI_14, dtype: float64

Missing Values: 140


,Date,Ticker,Adj Close,RSI_14
63554,2026-10-09,TD.TO,164.190002,44.782280
63555,2026-10-13,TD.TO,167.830002,53.077633
63556,2026-10-14,TD.TO,167.570007,52.471280
63557,2026-10-15,TD.TO,170.020004,57.408913
63558,2026-10-16,TD.TO,170.259995,57.870614
63559,2026-10-19,TD.TO,169.610001,56.096910
63560,2026-10-20,TD.TO,171.710007,60.327567
63561,2026-10-21,TD.TO,175.110001,66.034391
63562,2026-10-22,TD.TO,175.139999,66.080753
63563,2026-10-23,TD.TO,174.470001,63.980250


In [20]:
# Calculate EMA 12 and EMA 26 for each ticker
ema_12 = daily_data.groupby("Ticker")["Adj Close"].transform(
    lambda x: x.ewm(
        span=12,
        adjust=False,
        min_periods=12
    ).mean()
)

ema_26 = daily_data.groupby("Ticker")["Adj Close"].transform(
    lambda x: x.ewm(
        span=26,
        adjust=False,
        min_periods=26
    ).mean()
)

# MACD Line
daily_data["MACD"] = ema_12 - ema_26

# Signal Line
daily_data["MACD_Signal"] = (
    daily_data.groupby("Ticker")["MACD"].transform(
        lambda x: x.ewm(
            span=9,
            adjust=False,
            min_periods=9
        ).mean()
    )
)

# MACD Histogram
daily_data["MACD_Histogram"] = (
    daily_data["MACD"] - daily_data["MACD_Signal"]
)

In [21]:
macd_columns = [
    "MACD",
    "MACD_Signal",
    "MACD_Histogram"
]

print("Missing Values:")
print(daily_data[macd_columns].isna().sum())

print("\nSummary Statistics:")
display(daily_data[macd_columns].describe())

display(
    daily_data[
        ["Date", "Ticker", "Adj Close"] + macd_columns
    ].tail(20)
)

Missing Values:
MACD              250
MACD_Signal       330
MACD_Histogram    330
dtype: int64

Summary Statistics:


,MACD,MACD_Signal,MACD_Histogram
count,63324.000000,63244.000000,63244.000000
mean,0.153335,0.152826,0.000692
std,1.114975,1.050730,0.334056
min,-16.912446,-15.094499,-5.673820
25%,-0.113405,-0.104152,-0.063727
50%,0.064864,0.061933,0.000222
75%,0.331913,0.319144,0.067709
max,14.296355,11.995421,4.865464


,Date,Ticker,Adj Close,MACD,MACD_Signal,MACD_Histogram
63554,2026-10-09,TD.TO,164.190002,-0.943061,-0.255457,-0.687604
63555,2026-10-13,TD.TO,167.830002,-0.677083,-0.339782,-0.337301
63556,2026-10-14,TD.TO,167.570007,-0.481720,-0.368170,-0.113550
63557,2026-10-15,TD.TO,170.020004,-0.127727,-0.320081,0.192354
63558,2026-10-16,TD.TO,170.259995,0.170218,-0.222021,0.392239
63559,2026-10-19,TD.TO,169.610001,0.349860,-0.107645,0.457505
63560,2026-10-20,TD.TO,171.710007,0.654139,0.044712,0.609428
63561,2026-10-21,TD.TO,175.110001,1.156305,0.267030,0.889275
63562,2026-10-22,TD.TO,175.139999,1.538956,0.521416,1.017540
63563,2026-10-23,TD.TO,174.470001,1.767768,0.770686,0.997082


### Feature Engineering

The feature engineering phase focused on transforming historical stock price data into meaningful technical indicators that could help machine learning models identify patterns in stock price movements and predict future returns.

All price-based features were calculated independently for each stock (Ticker) using historical data, ensuring that information from different stocks was not mixed and that no future observations were used.

The following categories of features were created:

**1\. Daily Returns**

Calculated the daily percentage change in adjusted closing prices to measure the daily performance of each stock.

**2\. Lagged Returns**

Created historical return features over different time lags to capture previous price movements and provide the model with information about recent market behavior.

**3\. Rolling Volatility**

Calculated rolling standard deviations of daily returns over different windows to measure changes in stock price volatility and market risk.

**4\. Moving Averages**

Calculated simple moving averages over multiple periods to identify short-term and long-term price trends.

**5\. Moving Average Ratios**

Measured the percentage difference between the current adjusted closing price and its moving averages. These features help indicate whether a stock is trading above or below its recent average price.

**6\. Momentum Indicators**

Created momentum features over multiple periods to measure the cumulative percentage change in stock prices and capture the strength and direction of recent price movements.

**7\. Relative Strength Index (RSI)**

Added the 14-period RSI using Wilder's smoothing method to measure the strength of recent price movements and identify potentially overbought or oversold conditions.

**8\. MACD Indicators**

Calculated the MACD line, Signal line, and Histogram using standard exponential moving average settings. These indicators help capture changes in price trends and momentum.

Overall, this phase produced a set of technical features representing different aspects of stock market behavior, including returns, trends, volatility, and momentum. These features will be reviewed for missing values, correlation, and redundancy before being used in model training.

### Feature Engineering Summary

Created technical indicators from historical stock prices to capture market trends, returns, volatility, and momentum.

The engineered features include daily and lagged returns, rolling volatility, moving averages, moving average ratios, momentum indicators, RSI, and MACD.

All calculations were performed separately for each stock using historical observations only, providing the foundation for the next stages of feature selection and machine learning model development.

# Macroeconomic Feature Engineering

In [23]:
monthly_data = pd.read_csv(
    DATA_DIR / "canada_macro_monthly_combined.csv",
    parse_dates=["Date", "Availability_Date"]
)

print("Shape:", monthly_data.shape)

print("\nAvailability lag distribution:")
print(
    (monthly_data["Availability_Date"] - monthly_data["Date"])
    .dt.days.value_counts()
    .sort_index()
)

display(
    monthly_data[
        ["Date", "Availability_Date", "CPI",
         "GDP_Real_Millions_CAD", "BCPI_Total"]
    ].tail(15)
)

Shape: (1354, 18)

Availability lag distribution:
59    170
60     56
61    903
62    225
Name: count, dtype: int64


,Date,Availability_Date,CPI,GDP_Real_Millions_CAD,BCPI_Total
1339,2025-08-01,2025-10-01,164.8,2336966.0,601.47
1340,2025-09-01,2025-11-01,164.9,2342644.0,605.45
1341,2025-10-01,2025-12-01,165.3,2335762.0,602.66
1342,2025-11-01,2026-01-01,165.4,2337816.0,609.84
1343,2025-12-01,2026-02-01,165.0,2340246.0,604.64
1344,2026-01-01,2026-03-01,165.0,2339783.0,628.73
1345,2026-02-01,2026-04-01,165.9,2343599.0,643.36
1346,2026-03-01,2026-05-01,167.4,2340035.0,754.52
1347,2026-04-01,2026-06-01,168.0,2354837.0,765.04
1348,2026-05-01,2026-07-01,169.6,2362878.0,775.57


In [24]:
print("Macro availability lag in daily data:")

print(
    (
        daily_data["Macro_Availability_Date"]
        - daily_data["Macro_Reference_Date"]
    ).dt.days.value_counts().sort_index()
)

display(
    daily_data[
        ["Date", "Macro_Reference_Date", "Macro_Availability_Date"]
    ].drop_duplicates().tail(15)
)

Macro availability lag in daily data:
89     3993
90     9385
91    13122
92    37074
Name: count, dtype: int64


,Date,Macro_Reference_Date,Macro_Availability_Date
6729,2026-10-19,2026-07-01,2026-10-01
6730,2026-10-20,2026-07-01,2026-10-01
6731,2026-10-21,2026-07-01,2026-10-01
6732,2026-10-22,2026-07-01,2026-10-01
6733,2026-10-23,2026-07-01,2026-10-01
6734,2026-10-26,2026-07-01,2026-10-01
6735,2026-10-27,2026-07-01,2026-10-01
6736,2026-10-28,2026-07-01,2026-10-01
6737,2026-10-29,2026-07-01,2026-10-01
6738,2026-10-30,2026-07-01,2026-10-01


In [25]:
# Selected commodity indicators
commodity_features = {
    "BCPI_Total": "BCPI_Change_12M",
    "BCPI_Energy": "BCPI_Energy_Change_12M",
    "BCPI_Metals": "BCPI_Metals_Change_12M"
}

# Calculate year-over-year percentage changes
for source, new_feature in commodity_features.items():
    monthly_data[new_feature] = (
        monthly_data[source].pct_change(periods=12) * 100
    )

In [26]:
new_features = list(commodity_features.values())

print("Missing Values:")
print(monthly_data[new_features].isna().sum())

print("\nSummary Statistics:")
display(monthly_data[new_features].describe())

display(
    monthly_data[
        ["Date", "BCPI_Total", "BCPI_Energy", "BCPI_Metals"]
        + new_features
    ].tail(15)
)

Missing Values:
BCPI_Change_12M           708
BCPI_Energy_Change_12M    708
BCPI_Metals_Change_12M    708
dtype: int64

Summary Statistics:


,BCPI_Change_12M,BCPI_Energy_Change_12M,BCPI_Metals_Change_12M
count,646.000000,646.000000,646.000000
mean,5.391503,9.875378,6.187355
std,19.546453,33.634325,18.830686
min,-48.838306,-75.467877,-29.354974
25%,-4.729023,-9.357946,-7.927483
50%,3.309108,5.863713,3.280319
75%,14.742891,25.016195,16.672269
max,141.608714,308.163979,75.346042


,Date,BCPI_Total,BCPI_Energy,BCPI_Metals,BCPI_Change_12M,BCPI_Energy_Change_12M,BCPI_Metals_Change_12M
1339,2025-08-01,601.47,1175.97,948.11,-0.255385,-13.471175,19.021077
1340,2025-09-01,605.45,1185.45,989.14,3.765339,-7.120416,22.262462
1341,2025-10-01,602.66,1148.28,1045.25,-0.419696,-13.850355,25.155659
1342,2025-11-01,609.84,1186.32,1053.69,1.470882,-9.884918,27.795903
1343,2025-12-01,604.64,1144.47,1092.69,-0.023149,-13.980022,32.719146
1344,2026-01-01,628.73,1177.62,1179.46,0.466595,-15.610018,40.960644
1345,2026-02-01,643.36,1205.79,1204.32,3.558954,-9.788797,37.644437
1346,2026-03-01,754.52,1638.05,1204.76,21.104923,23.902273,34.271003
1347,2026-04-01,765.04,1672.05,1219.97,27.400500,37.889659,33.984602
1348,2026-05-01,775.57,1716.85,1217.14,29.570476,45.126796,31.013326


In [27]:
# Labour market feature engineering

monthly_data["Unemployment_Change_12M"] = (
    monthly_data["Unemployment_Rate"]
    .diff(periods=12)
)

monthly_data["Employment_Growth_12M"] = (
    monthly_data["Employment"]
    .pct_change(periods=12) * 100
)

In [28]:
labour_features = [
    "Unemployment_Change_12M",
    "Employment_Growth_12M"
]

print("Missing Values:")
print(monthly_data[labour_features].isna().sum())

print("\nSummary Statistics:")
display(monthly_data[labour_features].describe())

display(
    monthly_data[
        [
            "Date",
            "Employment",
            "Unemployment_Rate",
            "Unemployment_Change_12M",
            "Employment_Growth_12M"
        ]
    ].tail(15)
)

Missing Values:
Unemployment_Change_12M    756
Employment_Growth_12M      756
dtype: int64

Summary Statistics:


,Unemployment_Change_12M,Employment_Growth_12M
count,598.000000,598.000000
mean,-0.009866,1.585614
std,1.354892,2.151727
min,-5.900000,-15.576502
25%,-0.700000,0.951162
50%,-0.200000,1.836439
75%,0.400000,2.578342
max,8.700000,16.088724


,Date,Employment,Unemployment_Rate,Unemployment_Change_12M,Employment_Growth_12M
1339,2025-08-01,20956.6,7.1,0.5,0.945068
1340,2025-09-01,21009.9,7.1,0.6,1.028563
1341,2025-10-01,21083.6,6.9,0.3,1.339588
1342,2025-11-01,21135.9,6.6,-0.4,1.410133
1343,2025-12-01,21146.0,6.8,0.1,1.009329
1344,2026-01-01,21121.2,6.5,-0.2,0.639923
1345,2026-02-01,21037.3,6.7,0.1,0.246837
1346,2026-03-01,21051.4,6.7,-0.1,0.415468
1347,2026-04-01,21033.7,6.9,0.0,0.319554
1348,2026-05-01,21121.5,6.6,-0.4,0.701331


In [29]:
# Yield spread
daily_data["Yield_Spread"] = (
    daily_data["Bond_Yield_10Y"] - daily_data["Policy_Rate"]
)

# Policy rate change over approximately one trading month
daily_data["Policy_Rate_Change_21D"] = (
    daily_data.groupby("Ticker")["Policy_Rate"]
    .diff(periods=21)
)

In [30]:
interest_features = [
    "Yield_Spread",
    "Policy_Rate_Change_21D"
]

print("Missing Values:")
print(daily_data[interest_features].isna().sum())

print("\nSummary Statistics:")
display(daily_data[interest_features].describe())

display(
    daily_data[
        [
            "Date",
            "Ticker",
            "Policy_Rate",
            "Bond_Yield_10Y",
            "Yield_Spread",
            "Policy_Rate_Change_21D"
        ]
    ].tail(20)
)

Missing Values:
Yield_Spread              21294
Policy_Rate_Change_21D    21320
dtype: int64

Summary Statistics:


,Yield_Spread,Policy_Rate_Change_21D
count,42280.000000,42254.000000
mean,0.802406,0.009810
std,1.109566,0.168923
min,-1.940000,-1.500000
25%,0.320000,0.000000
50%,0.940000,0.000000
75%,1.290000,0.000000
max,3.470000,1.000000


,Date,Ticker,Policy_Rate,Bond_Yield_10Y,Yield_Spread,Policy_Rate_Change_21D
63554,2026-10-09,TD.TO,2.25,3.86,1.61,0.00
63555,2026-10-13,TD.TO,2.25,3.87,1.62,0.00
63556,2026-10-14,TD.TO,2.25,3.85,1.60,0.00
63557,2026-10-15,TD.TO,2.25,3.83,1.58,0.00
63558,2026-10-16,TD.TO,2.25,3.84,1.59,0.00
63559,2026-10-19,TD.TO,2.25,3.93,1.68,0.00
63560,2026-10-20,TD.TO,2.25,3.90,1.65,0.00
63561,2026-10-21,TD.TO,2.25,3.90,1.65,0.00
63562,2026-10-22,TD.TO,2.25,3.78,1.53,0.00
63563,2026-10-23,TD.TO,2.25,3.83,1.58,0.00


In [35]:
daily_data["USD_CAD_Change_21D"] = (
    daily_data.groupby("Ticker")["USD_CAD"]
    .transform(
        lambda x: x.pct_change(
            periods=21,
            fill_method=None
        ) * 100
    )
)

In [36]:
print(daily_data["USD_CAD_Change_21D"].isna().sum())

display(
    daily_data["USD_CAD_Change_21D"].describe()
)

39324


count    24250.000000
mean         0.067589
std          1.825518
min         -5.735604
25%         -1.142857
50%          0.059180
75%          1.228916
max          9.553322
Name: USD_CAD_Change_21D, dtype: float64

In [37]:
monthly_data.to_csv(
    DATA_DIR / "canada_macro_monthly_features.csv",
    index=False
)

In [38]:
daily_data.to_csv(
    DATA_DIR / "canada_features_engineered.csv",
    index=False
)

# Macroeconomic Feature Engineering

## 1. Overview

In this stage, additional macroeconomic features were engineered from the previously integrated Canadian economic datasets. The objective was to capture changes in commodity prices, labour market conditions, interest rates, and currency exchange rates that may provide useful information for analyzing and predicting Canadian stock market movements.

Feature engineering was performed using the monthly macroeconomic dataset (monthly_data) and the integrated daily dataset (daily_data), depending on the frequency of each indicator.

## 2. Commodity Market Features

Three annual percentage-change features were created using the monthly Bank of Canada Commodity Price Index (BCPI):

- BCPI_Change_12M
- BCPI_Energy_Change_12M
- BCPI_Metals_Change_12M

These features measure the percentage change in the total commodity index, energy index, and metals index compared with the same month of the previous year.

**Key findings:**

- Total BCPI annual change averaged approximately 5.39%.
- Energy BCPI annual change averaged approximately 9.88%.
- Metals BCPI annual change averaged approximately 6.19%.
- In October 2026, the annual changes were approximately 21.39%, 41.09%, and 5.58%, respectively.

These features help represent changes in commodity market conditions, particularly those relevant to Canada's resource-oriented economy.

## 3. Labour Market Features

Two features were engineered from Statistics Canada's monthly Labour Force Survey data:

- Unemployment_Change_12M: Measures the difference in the unemployment rate compared with the same month one year earlier, expressed in percentage points.
- Employment_Growth_12M: Measures the annual percentage growth in employment.

**Key findings:**

- Average unemployment-rate change was approximately -0.01 percentage points.
- Average annual employment growth was approximately 1.59%.
- In October 2026, unemployment was 0.2 percentage points higher than the previous year, while employment growth was approximately 0.08%.

These features capture changes in employment conditions and labour market trends.

## 4. Interest Rate Features

Two features were created using daily Bank of Canada indicators:

- Yield_Spread: Calculated as the 10-year government bond yield minus the policy interest rate.
- Policy_Rate_Change_21D: Measures the change in the policy rate over 21 previous records.

**Key findings:**

- Average yield spread was approximately 0.80 percentage points.
- Average policy-rate change over 21 records was approximately 0.01 percentage points.
- The median policy-rate change was zero, reflecting the fact that policy rates generally remain unchanged between monetary policy decisions.

These features describe the relationship between short-term policy rates and long-term bond yields, as well as changes in monetary policy.

## 5. Currency Exchange Rate Feature

One feature was created using the daily USD/CAD exchange rate:

- USD_CAD_Change_21D: Measures the percentage change in the USD/CAD exchange rate over 21 previous records.

**Key findings:**

- Mean change: approximately 0.07%.
- Standard deviation: approximately 1.82%.
- Minimum: approximately -5.74%.
- Maximum: approximately 9.55%.

A positive value indicates an increase in USD/CAD, meaning the US dollar strengthened relative to the Canadian dollar. A negative value indicates the opposite.

This feature provides information about currency movements that may affect Canadian companies, particularly those exposed to international trade and foreign exchange fluctuations.

## 6. Missing Values and Data Considerations

Missing values were observed across several engineered features due to differences in historical data coverage, unavailable observations, and the lookback periods required for calculating changes.

The missing values were not automatically replaced with zeros or removed. This preserves the original information and avoids introducing artificial observations.

The current macroeconomic availability dates also use assumed publication lags. These are temporary approximations and must be verified against official release dates before final model evaluation to reduce the risk of look-ahead bias.

## 7. Conclusion

This stage produced eight additional macroeconomic features across four categories: commodities, labour market, interest rates, and currency exchange rates.

These features expand the dataset beyond individual stock-price indicators by incorporating broader economic conditions that may influence Canadian equity markets.

The next stage will focus on calendar-based features, followed by a comprehensive feature review, missing-value assessment, and preparation of the final modeling dataset.

# Macroeconomic Feature Engineering – Summary

Engineered eight additional macroeconomic features using monthly Statistics Canada data and daily Bank of Canada indicators.

- **Commodities:** Created 12-month percentage-change features for total BCPI, energy, and metals indices.
- **Labour Market:** Created annual unemployment-rate change and employment-growth features.
- **Interest Rates:** Calculated the 10-year bond yield spread over the policy rate and the 21-record policy-rate change.
- **Currency:** Created a 21-record percentage-change feature for the USD/CAD exchange rate.

These features capture changes in Canada's commodity markets, employment conditions, monetary policy, and currency movements.

Missing values were retained for further assessment rather than automatically filled or removed. Publication-date assumptions and potential look-ahead bias will also be reviewed before model training.

**Outcome:** Eight new macroeconomic features were engineered to complement the existing stock-market technical indicators.

# Calendar Features

In [5]:
import numpy as np

# Extract calendar components
daily_data["DayOfWeek"] = daily_data["Date"].dt.dayofweek
daily_data["Month"] = daily_data["Date"].dt.month
daily_data["Quarter"] = daily_data["Date"].dt.quarter

# Cyclical encoding for weekdays
daily_data["DayOfWeek_Sin"] = np.sin(
    2 * np.pi * daily_data["DayOfWeek"] / 5
)

daily_data["DayOfWeek_Cos"] = np.cos(
    2 * np.pi * daily_data["DayOfWeek"] / 5
)

# Cyclical encoding for months
daily_data["Month_Sin"] = np.sin(
    2 * np.pi * (daily_data["Month"] - 1) / 12
)

daily_data["Month_Cos"] = np.cos(
    2 * np.pi * (daily_data["Month"] - 1) / 12
)

In [6]:
calendar_features = [
    "DayOfWeek",
    "Month",
    "Quarter",
    "DayOfWeek_Sin",
    "DayOfWeek_Cos",
    "Month_Sin",
    "Month_Cos"
]

print("Missing Values:")
print(daily_data[calendar_features].isna().sum())

print("\nSummary Statistics:")
display(daily_data[calendar_features].describe())

display(
    daily_data[
        ["Date", "Ticker"] + calendar_features
    ].tail(15)
)

Missing Values:
DayOfWeek        0
Month            0
Quarter          0
DayOfWeek_Sin    0
DayOfWeek_Cos    0
Month_Sin        0
Month_Cos        0
dtype: int64

Summary Statistics:


,DayOfWeek,Month,Quarter,DayOfWeek_Sin,DayOfWeek_Cos,Month_Sin,Month_Cos
count,63574.000000,63574.000000,63574.000000,63574.000000,63574.000000,6.357400e+04,6.357400e+04
mean,2.038129,6.474313,2.493000,0.002355,-0.023642,4.396692e-03,-1.155556e-02
std,1.395704,3.424161,1.113919,0.713566,0.700196,7.084044e-01,7.057096e-01
min,0.000000,1.000000,1.000000,-0.951057,-0.809017,-1.000000e+00,-1.000000e+00
25%,1.000000,4.000000,2.000000,-0.587785,-0.809017,-8.660254e-01,-8.660254e-01
50%,2.000000,6.000000,2.000000,0.000000,0.309017,1.224647e-16,-1.836970e-16
75%,3.000000,9.000000,3.000000,0.587785,0.309017,8.660254e-01,5.000000e-01
max,4.000000,12.000000,4.000000,0.951057,1.000000,1.000000e+00,1.000000e+00


,Date,Ticker,DayOfWeek,Month,Quarter,DayOfWeek_Sin,DayOfWeek_Cos,Month_Sin,Month_Cos
63559,2026-10-19,TD.TO,0,10,4,0.000000,1.000000,-1.000000,-1.836970e-16
63560,2026-10-20,TD.TO,1,10,4,0.951057,0.309017,-1.000000,-1.836970e-16
63561,2026-10-21,TD.TO,2,10,4,0.587785,-0.809017,-1.000000,-1.836970e-16
63562,2026-10-22,TD.TO,3,10,4,-0.587785,-0.809017,-1.000000,-1.836970e-16
63563,2026-10-23,TD.TO,4,10,4,-0.951057,0.309017,-1.000000,-1.836970e-16
63564,2026-10-26,TD.TO,0,10,4,0.000000,1.000000,-1.000000,-1.836970e-16
63565,2026-10-27,TD.TO,1,10,4,0.951057,0.309017,-1.000000,-1.836970e-16
63566,2026-10-28,TD.TO,2,10,4,0.587785,-0.809017,-1.000000,-1.836970e-16
63567,2026-10-29,TD.TO,3,10,4,-0.587785,-0.809017,-1.000000,-1.836970e-16
63568,2026-10-30,TD.TO,4,10,4,-0.951057,0.309017,-1.000000,-1.836970e-16


In [7]:
print("Day of Week Distribution:")
display(
    daily_data["DayOfWeek"]
    .value_counts()
    .sort_index()
)

print("Month Distribution:")
display(
    daily_data["Month"]
    .value_counts()
    .sort_index()
)

Day of Week Distribution:


DayOfWeek
0    11647
1    12992
2    13046
3    13068
4    12821
Name: count, dtype: int64

Month Distribution:


Month
1     5381
2     4945
3     5551
4     5249
5     5388
6     5460
7     5393
8     5392
9     5196
10    5401
11    5299
12    4919
Name: count, dtype: int64

In [8]:
daily_data.to_csv(
    DATA_DIR / "canada_daily_features_engineered.csv",
    index=False
)

# Feature Inventory & Missing Values Audit

In [3]:
DATA_DIR = Path("data/processed/main")

In [4]:
daily_data = pd.read_csv(
    DATA_DIR / "canada_daily_features_engineered.csv",
    parse_dates=[
        "Date",
        "Bank_Availability_Date",
        "Macro_Reference_Date",
        "Macro_Availability_Date"
    ]
)

daily_data = daily_data.sort_values(
    ["Ticker", "Date"]
).reset_index(drop=True)

print("Shape:", daily_data.shape)
print("Tickers:", daily_data["Ticker"].nunique())
print("Date Range:", daily_data["Date"].min(), "to", daily_data["Date"].max())

daily_data.head()

Shape: (63574, 39)
Tickers: 10
Date Range: 2000-01-04 00:00:00 to 2026-11-06 00:00:00


,Date,Ticker,Open,High,Low,Close,Adj Close,Volume,Dividends,Stock Splits,...,BCPI_Agriculture,BCPI_Fish,Macro_Availability_Date,DayOfWeek,Month,Quarter,DayOfWeek_Sin,DayOfWeek_Cos,Month_Sin,Month_Cos
0,2000-01-04,ABX.TO,25.150000,25.65,24.950001,25.450001,17.804482,926500,0.0,0.0,...,136.81,883.95,2000-01-01,1,1,1,0.951057,0.309017,0.0,1.0
1,2000-01-05,ABX.TO,25.400000,26.00,25.350000,25.650000,17.944401,1452200,0.0,0.0,...,136.81,883.95,2000-01-01,2,1,1,0.587785,-0.809017,0.0,1.0
2,2000-01-06,ABX.TO,25.799999,26.00,25.500000,25.799999,18.049345,600200,0.0,0.0,...,136.81,883.95,2000-01-01,3,1,1,-0.587785,-0.809017,0.0,1.0
3,2000-01-07,ABX.TO,25.850000,26.00,25.100000,25.250000,17.664574,1020800,0.0,0.0,...,136.81,883.95,2000-01-01,4,1,1,-0.951057,0.309017,0.0,1.0
4,2000-01-10,ABX.TO,25.350000,25.35,25.150000,25.250000,17.664574,1281100,0.0,0.0,...,136.81,883.95,2000-01-01,0,1,1,0.000000,1.000000,0.0,1.0


In [5]:
print("Dataset Shape:", daily_data.shape)

print("\nTotal Columns:", len(daily_data.columns))

print("\nColumn Names:")
for i, col in enumerate(daily_data.columns, 1):
    print(f"{i}. {col}")

Dataset Shape: (63574, 39)

Total Columns: 39

Column Names:
1. Date
2. Ticker
3. Open
4. High
5. Low
6. Close
7. Adj Close
8. Volume
9. Dividends
10. Stock Splits
11. Bank_Availability_Date
12. Policy_Rate
13. Bond_Yield_10Y
14. USD_CAD
15. Macro_Reference_Date
16. CPI
17. Inflation_YoY
18. GDP_Real_Millions_CAD
19. GDP_MoM
20. GDP_YoY
21. Employment
22. Labour_Force
23. Participation_Rate
24. Unemployment_Rate
25. BCPI_Total
26. BCPI_NonEnergy
27. BCPI_Energy
28. BCPI_Metals
29. BCPI_Forestry
30. BCPI_Agriculture
31. BCPI_Fish
32. Macro_Availability_Date
33. DayOfWeek
34. Month
35. Quarter
36. DayOfWeek_Sin
37. DayOfWeek_Cos
38. Month_Sin
39. Month_Cos


In [6]:
missing_summary = pd.DataFrame({
    "Missing_Count": daily_data.isna().sum(),
    "Missing_Percentage": (
        daily_data.isna().mean() * 100
    )
})

missing_summary = missing_summary[
    missing_summary["Missing_Count"] > 0
].sort_values(
    by="Missing_Percentage",
    ascending=False
)

display(missing_summary)

,Missing_Count,Missing_Percentage
USD_CAD,38984,61.320666
Policy_Rate,21090,33.173939
Bond_Yield_10Y,2473,3.889955
Bank_Availability_Date,2259,3.553339


In [7]:
print(
    "Duplicate Date-Ticker:",
    daily_data.duplicated(
        subset=["Date", "Ticker"]
    ).sum()
)

print(
    "Duplicate Rows:",
    daily_data.duplicated().sum()
)

Duplicate Date-Ticker: 0
Duplicate Rows: 0


In [8]:
monthly_features = pd.read_csv(
    DATA_DIR / "canada_macro_monthly_features.csv",
    parse_dates=["Date", "Availability_Date"]
)

print("Shape:", monthly_features.shape)
print("\nColumns:")
print(monthly_features.columns.tolist())

Shape: (1354, 23)

Columns:
['Date', 'CPI', 'Inflation_YoY', 'GDP_Real_Millions_CAD', 'GDP_MoM', 'GDP_YoY', 'Employment', 'Labour_Force', 'Participation_Rate', 'Unemployment_Rate', 'BCPI_Total', 'BCPI_NonEnergy', 'BCPI_Energy', 'BCPI_Metals', 'BCPI_Forestry', 'BCPI_Agriculture', 'BCPI_Fish', 'Availability_Date', 'BCPI_Change_12M', 'BCPI_Energy_Change_12M', 'BCPI_Metals_Change_12M', 'Unemployment_Change_12M', 'Employment_Growth_12M']


## Feature Inventory & Missing Values Audit

An initial feature inventory and missing values audit was conducted on the integrated Canadian stock market dataset to assess its structure, identify available variables, and evaluate data completeness before proceeding with feature engineering.

The dataset contained **63,574 records and 39 columns**, covering 10 Canadian stocks over the period from January 4, 2000, to November 6, 2026.

The available variables were organized into the following categories:

- **Stock Market Data:** Open, High, Low, Close, Adjusted Close, Volume, Dividends, and Stock Splits.
- **Daily Economic Indicators:** Bank of Canada Policy Rate, 10-Year Bond Yield, and USD/CAD Exchange Rate.
- **Macroeconomic Indicators:** CPI, Inflation, GDP, Employment, Labour Force, Participation Rate, Unemployment Rate, and commodity price indices.
- **Calendar Features:** Day of Week, Month, Quarter, and their cyclical sine/cosine transformations.

The missing values audit identified the following:

| Feature                | Missing Values | Missing Percentage |
| ---------------------- | -------------- | ------------------ |
| USD_CAD                | 38,984         | 61.32%             |
| Policy_Rate            | 21,090         | 33.17%             |
| Bond_Yield_10Y         | 2,473          | 3.89%              |
| Bank_Availability_Date | 2,259          | 3.55%              |

Most missing values were associated with differences in historical coverage and observation frequency among the economic indicators, rather than errors in the stock market records.

The structural integrity checks also confirmed:

- No duplicate Date-Ticker combinations.
- No completely duplicated records.
- No missing values in the calendar features.
- Consistent date ordering within each ticker.

The audit established a baseline for subsequent feature engineering and highlighted the need to handle missing economic observations carefully without introducing artificial values or future information.

No missing values were removed or arbitrarily replaced at this stage.

## Feature Inventory & Missing Values Audit

The integrated dataset contained **63,574 records and 39 columns**, covering stock prices, daily economic indicators, monthly macroeconomic data, and calendar features. The missing values audit identified higher missingness in USD/CAD (61.32%) and the Policy Rate (33.17%), mainly due to differences in historical data coverage. No duplicate records or Date-Ticker combinations were found, and no missing values were detected in the calendar features. The dataset was preserved without arbitrary imputation or deletion.

# Calendar Features – Detailed Report

## 1. Overview

In this stage, calendar-based features were extracted from the Date column of the integrated daily dataset. The objective was to capture potential temporal patterns in stock market behavior, including differences across weekdays, months, and quarters.

## 2. Feature Creation

Seven calendar features were created:

| Feature       | Description                                            |
| ------------- | ------------------------------------------------------ |
| DayOfWeek     | Represents the weekday, from Monday (0) to Friday (4). |
| Month         | Represents the calendar month, from 1 to 12.           |
| Quarter       | Represents the calendar quarter, from 1 to 4.          |
| DayOfWeek_Sin | Sine transformation of the weekday.                    |
| DayOfWeek_Cos | Cosine transformation of the weekday.                  |
| Month_Sin     | Sine transformation of the month.                      |
| Month_Cos     | Cosine transformation of the month.                    |

Cyclical encoding was applied to weekdays and months to preserve their circular relationships and avoid treating the end and beginning of each cycle as distant values.

## 3. Results and Observations

All seven features were successfully generated across 63,574 records, with no missing values.

The weekday distribution was relatively balanced, ranging from 11,647 Monday records to 13,068 Thursday records.

Monthly records were also distributed across all twelve months, ranging from 4,919 in December to 5,551 in March.

The cyclical features remained within their expected range of -1 to 1.

## 4. Conclusion

Calendar feature engineering was completed successfully. These features provide temporal information that may help machine learning models identify weekday patterns, monthly seasonality, and quarterly effects in Canadian stock market data.

The next step is a comprehensive review of all engineered features, including missing values, feature redundancy, data consistency, and potential target leakage before preparing the final modeling dataset.

# Calendar Features – Summary

Created seven calendar features from the Date column, including weekday, month, quarter, and cyclical sine/cosine transformations for weekdays and months.

All 63,574 records were successfully processed with no missing values. Weekday and monthly distributions were relatively balanced, covering all five trading days and twelve calendar months.

**Outcome:** Calendar features were successfully engineered to capture potential temporal patterns and seasonality in Canadian stock market data.

# Monthly-to-Daily Feature Integration

In [9]:
# Monthly features to integrate
macro_features = [
    "BCPI_Change_12M",
    "BCPI_Energy_Change_12M",
    "BCPI_Metals_Change_12M",
    "Unemployment_Change_12M",
    "Employment_Growth_12M"
]

# Create a separate availability date using the current 3-month assumption
monthly_features_data = monthly_features[
    ["Date"] + macro_features
].copy(deep=True)

monthly_features_data["Macro_Feature_Availability_Date"] = (
    monthly_features_data["Date"] + pd.DateOffset(months=3)
)

# Keep only the required columns
monthly_features_data = monthly_features_data[
    ["Macro_Feature_Availability_Date"] + macro_features
]

# Sort both datasets by their merge keys
daily_sorted = daily_data.sort_values(
    ["Date", "Ticker"]
).copy(deep=True)

monthly_sorted = monthly_features_data.sort_values(
    "Macro_Feature_Availability_Date"
).copy(deep=True)

# Remove previously merged versions if this cell is being rerun
# remove from integrated data between bank and stocks
daily_sorted = daily_sorted.drop(
    columns=macro_features + ["Macro_Feature_Availability_Date"],
    errors="ignore"
)

# Merge using the latest available monthly observation
daily_data = pd.merge_asof(
    daily_sorted,
    monthly_sorted,
    left_on="Date",
    right_on="Macro_Feature_Availability_Date",
    direction="backward"
)

# Restore ticker/date ordering
daily_data = daily_data.sort_values(
    ["Ticker", "Date"]
).reset_index(drop=True)

In [10]:
print("Dataset Shape:", daily_data.shape)

print("\nMissing Values:")
display(daily_data[macro_features].isna().sum())

print("\nDuplicate Date-Ticker:")
print(
    daily_data.duplicated(
        subset=["Date", "Ticker"]
    ).sum()
)

print("\nLatest Records:")
display(
    daily_data[
        [
            "Date",
            "Ticker",
            "Macro_Feature_Availability_Date"
        ] + macro_features
    ].tail(15)
)

Dataset Shape: (63574, 45)

Missing Values:


BCPI_Change_12M            0
BCPI_Energy_Change_12M     0
BCPI_Metals_Change_12M     0
Unemployment_Change_12M    0
Employment_Growth_12M      0
dtype: int64


Duplicate Date-Ticker:
0

Latest Records:


,Date,Ticker,Macro_Feature_Availability_Date,BCPI_Change_12M,BCPI_Energy_Change_12M,BCPI_Metals_Change_12M,Unemployment_Change_12M,Employment_Growth_12M
63559,2026-10-19,TD.TO,2026-10-01,11.807769,13.815136,17.973925,-0.5,0.934900
63560,2026-10-20,TD.TO,2026-10-01,11.807769,13.815136,17.973925,-0.5,0.934900
63561,2026-10-21,TD.TO,2026-10-01,11.807769,13.815136,17.973925,-0.5,0.934900
63562,2026-10-22,TD.TO,2026-10-01,11.807769,13.815136,17.973925,-0.5,0.934900
63563,2026-10-23,TD.TO,2026-10-01,11.807769,13.815136,17.973925,-0.5,0.934900
63564,2026-10-26,TD.TO,2026-10-01,11.807769,13.815136,17.973925,-0.5,0.934900
63565,2026-10-27,TD.TO,2026-10-01,11.807769,13.815136,17.973925,-0.5,0.934900
63566,2026-10-28,TD.TO,2026-10-01,11.807769,13.815136,17.973925,-0.5,0.934900
63567,2026-10-29,TD.TO,2026-10-01,11.807769,13.815136,17.973925,-0.5,0.934900
63568,2026-10-30,TD.TO,2026-10-01,11.807769,13.815136,17.973925,-0.5,0.934900


In [11]:
# Compare the two availability dates
mismatch = (
    daily_data["Macro_Feature_Availability_Date"]
    != daily_data["Macro_Availability_Date"]
)

print("Availability Date Mismatches:", mismatch.sum())

print("\nSample Audit:")
display(
    daily_data[
        [
            "Date",
            "Macro_Reference_Date",
            "Macro_Availability_Date",
            "Macro_Feature_Availability_Date"
        ]
    ].tail(15)
)

Availability Date Mismatches: 0

Sample Audit:


,Date,Macro_Reference_Date,Macro_Availability_Date,Macro_Feature_Availability_Date
63559,2026-10-19,2026-07-01,2026-10-01,2026-10-01
63560,2026-10-20,2026-07-01,2026-10-01,2026-10-01
63561,2026-10-21,2026-07-01,2026-10-01,2026-10-01
63562,2026-10-22,2026-07-01,2026-10-01,2026-10-01
63563,2026-10-23,2026-07-01,2026-10-01,2026-10-01
63564,2026-10-26,2026-07-01,2026-10-01,2026-10-01
63565,2026-10-27,2026-07-01,2026-10-01,2026-10-01
63566,2026-10-28,2026-07-01,2026-10-01,2026-10-01
63567,2026-10-29,2026-07-01,2026-10-01,2026-10-01
63568,2026-10-30,2026-07-01,2026-10-01,2026-10-01


In [12]:
output_path = DATA_DIR / "canada_integrated_daily_macro_features.csv"

daily_data.to_csv(output_path, index=False)

print("Saved successfully:", output_path)

Saved successfully: data/processed/main/canada_integrated_daily_macro_features.csv


In [13]:
print("Shape:", daily_data.shape)
print("\nColumns:")
print(daily_data.columns.tolist())

Shape: (63574, 45)

Columns:
['Date', 'Ticker', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'Dividends', 'Stock Splits', 'Bank_Availability_Date', 'Policy_Rate', 'Bond_Yield_10Y', 'USD_CAD', 'Macro_Reference_Date', 'CPI', 'Inflation_YoY', 'GDP_Real_Millions_CAD', 'GDP_MoM', 'GDP_YoY', 'Employment', 'Labour_Force', 'Participation_Rate', 'Unemployment_Rate', 'BCPI_Total', 'BCPI_NonEnergy', 'BCPI_Energy', 'BCPI_Metals', 'BCPI_Forestry', 'BCPI_Agriculture', 'BCPI_Fish', 'Macro_Availability_Date', 'DayOfWeek', 'Month', 'Quarter', 'DayOfWeek_Sin', 'DayOfWeek_Cos', 'Month_Sin', 'Month_Cos', 'Macro_Feature_Availability_Date', 'BCPI_Change_12M', 'BCPI_Energy_Change_12M', 'BCPI_Metals_Change_12M', 'Unemployment_Change_12M', 'Employment_Growth_12M']


## Monthly-to-Daily Feature Integration

The engineered monthly macroeconomic features were integrated into the daily stock market dataset to provide additional economic context for each stock observation.

The monthly engineered dataset, canada_macro_monthly_features.csv, contained **1,354 records and 23 columns**, including the original macroeconomic indicators and five newly engineered features:

**Commodity Market Features:**

- BCPI_Change_12M: 12-month percentage change in the total Bank of Canada Commodity Price Index.
- BCPI_Energy_Change_12M: 12-month percentage change in the energy commodity index.
- BCPI_Metals_Change_12M: 12-month percentage change in the metals commodity index.

**Labour Market Features:**

- Unemployment_Change_12M: Change in the unemployment rate over 12 months, measured in percentage points.
- Employment_Growth_12M: 12-month percentage growth in employment.

### Integration Methodology

Since macroeconomic indicators are published monthly while stock prices are recorded daily, a time-aware integration approach was required.

The merge_asof() function was used with:

- direction="backward" to select the latest available observation.
- A separate macro feature availability date.
- A temporary three-month availability lag applied to the monthly reference date.

This approach ensured that each daily stock observation was matched with the latest macroeconomic feature whose assumed availability date was not later than the stock observation date.

The integration was performed using the availability date rather than the original monthly reference date to reduce the risk of look-ahead bias.

### Integration Results

The resulting daily dataset contained:

| Metric                         | Result |
| ------------------------------ | ------ |
| Records                        | 63,574 |
| Total Columns                  | 45     |
| Newly Integrated Features      | 5      |
| Missing Values in New Features | 0      |
| Duplicate Date-Ticker Records  | 0      |
| Availability Date Mismatches   | 0      |

A final availability-date audit confirmed that Macro_Feature_Availability_Date matched the existing Macro_Availability_Date for every record.

The resulting dataset successfully combined daily stock market information with engineered monthly economic features while preserving its original number of observations.

### Methodological Limitation

The three-month availability lag is a temporary modeling assumption rather than a verified publication schedule for each economic indicator. Actual release dates should be investigated before final model evaluation and historical backtesting.

The integrated dataset was preserved separately as canada_integrated_daily_macro_features.csv, keeping the original integrated dataset unchanged.

## Monthly-to-Daily Feature Integration

Five engineered monthly macroeconomic features covering commodity price changes and labour market trends were integrated into the daily stock dataset using merge_asof() with backward matching and a temporary three-month availability lag. The resulting dataset contained **63,574 records and 45 columns**, with no missing values in the newly integrated features and no duplicate Date-Ticker records. An availability-date audit confirmed zero mismatches with the original macroeconomic integration. The three-month lag remains a provisional assumption pending verification of actual publication dates.

# Daily Feature

In [14]:
daily_data = daily_data.sort_values(
    ["Ticker", "Date"]
).reset_index(drop=True)

In [15]:
daily_data["Daily_Return"] = (
    daily_data.groupby("Ticker")["Adj Close"]
    .pct_change(fill_method=None) * 100
)

In [16]:
lags = [1, 5, 10, 20]

for lag in lags:
    daily_data[f"Return_Lag_{lag}"] = (
        daily_data.groupby("Ticker")["Daily_Return"]
        .shift(lag)
    )

In [17]:
windows = [5, 10, 20]

for window in windows:
    daily_data[f"Volatility_{window}D"] = (
        daily_data.groupby("Ticker")["Daily_Return"]
        .transform(
            lambda x: x.rolling(window=window).std()
        )
    )

In [18]:
new_features = [
    "Daily_Return",
    "Return_Lag_1",
    "Return_Lag_5",
    "Return_Lag_10",
    "Return_Lag_20",
    "Volatility_5D",
    "Volatility_10D",
    "Volatility_20D"
]

print("Dataset Shape:", daily_data.shape)

print("\nMissing Values:")
display(daily_data[new_features].isna().sum())

print("\nSummary Statistics:")
display(daily_data[new_features].describe().T)

Dataset Shape: (63574, 53)

Missing Values:


Daily_Return       10
Return_Lag_1       20
Return_Lag_5       60
Return_Lag_10     110
Return_Lag_20     210
Volatility_5D      50
Volatility_10D    100
Volatility_20D    200
dtype: int64


Summary Statistics:


,count,mean,std,min,25%,50%,75%,max
Daily_Return,63564.0,0.073344,1.945237,-29.180444,-0.768343,0.074152,0.902516,32.462666
Return_Lag_1,63554.0,0.073313,1.945280,-29.180444,-0.768351,0.074191,0.902540,32.462666
Return_Lag_5,63514.0,0.073639,1.943896,-29.180444,-0.767959,0.074691,0.902493,32.462666
Return_Lag_10,63464.0,0.073685,1.944096,-29.180444,-0.768073,0.074135,0.902470,32.462666
Return_Lag_20,63364.0,0.073193,1.944351,-29.180444,-0.768343,0.073620,0.901371,32.462666
Volatility_5D,63524.0,1.526364,1.223695,0.042569,0.759427,1.215532,1.909031,16.576038
Volatility_10D,63474.0,1.592782,1.130265,0.164520,0.876234,1.316749,1.954135,14.875936
Volatility_20D,63374.0,1.631595,1.068515,0.237630,0.948040,1.378109,1.983919,13.951896


In [19]:
windows = [5, 10, 20, 50]

for window in windows:
    daily_data[f"MA_{window}"] = (
        daily_data.groupby("Ticker")["Adj Close"]
        .transform(
            lambda x: x.rolling(window=window).mean()
        )
    )

In [20]:
for window in windows:
    daily_data[f"MA_Ratio_{window}"] = (
        (daily_data["Adj Close"] / daily_data[f"MA_{window}"] - 1) * 100
    )

In [21]:
ma_features = [
    f"MA_{w}" for w in windows
] + [
    f"MA_Ratio_{w}" for w in windows
]

print("Dataset Shape:", daily_data.shape)

print("\nMissing Values:")
display(daily_data[ma_features].isna().sum())

print("\nSummary Statistics:")
display(daily_data[ma_features].describe().T)

Dataset Shape: (63574, 61)

Missing Values:


MA_5            40
MA_10           90
MA_20          190
MA_50          490
MA_Ratio_5      40
MA_Ratio_10     90
MA_Ratio_20    190
MA_Ratio_50    490
dtype: int64


Summary Statistics:


,count,mean,std,min,25%,50%,75%,max
MA_5,63534.0,37.229211,41.356337,0.691804,11.364168,22.994135,44.695743,298.291577
MA_10,63484.0,37.199050,41.270396,0.698218,11.379861,22.987450,44.600292,295.947473
MA_20,63384.0,37.138081,41.096044,0.713610,11.415289,22.983216,44.500867,294.318539
MA_50,63084.0,36.962904,40.598308,0.729217,11.454953,23.040669,44.525298,290.693279
MA_Ratio_5,63534.0,0.115518,2.092189,-28.822920,-0.822980,0.145794,1.072204,26.172879
MA_Ratio_10,63484.0,0.266748,3.161094,-45.759361,-1.208963,0.338003,1.776698,40.516060
MA_Ratio_20,63384.0,0.568592,4.498518,-60.936110,-1.605065,0.675170,2.808738,42.064603
MA_Ratio_50,63084.0,1.472361,6.969913,-68.689867,-2.095463,1.675057,5.044315,53.220533


In [22]:
momentum_periods = [5, 10, 20]

for period in momentum_periods:
    daily_data[f"Momentum_{period}D"] = (
        daily_data.groupby("Ticker")["Adj Close"]
        .transform(
            lambda x: (x / x.shift(period) - 1) * 100
        )
    )

In [23]:
momentum_features = [
    "Momentum_5D",
    "Momentum_10D",
    "Momentum_20D"
]

print("Dataset Shape:", daily_data.shape)

print("\nMissing Values:")
display(daily_data[momentum_features].isna().sum())

print("\nSummary Statistics:")
display(daily_data[momentum_features].describe().T)

Dataset Shape: (63574, 64)

Missing Values:


Momentum_5D      50
Momentum_10D    100
Momentum_20D    200
dtype: int64


Summary Statistics:


,count,mean,std,min,25%,50%,75%,max
Momentum_5D,63524.0,0.362051,4.238297,-47.013487,-1.666637,0.373576,2.314759,43.282065
Momentum_10D,63474.0,0.720491,5.853519,-66.787452,-2.184917,0.704987,3.509167,77.053640
Momentum_20D,63374.0,1.425676,8.004617,-72.772272,-2.722412,1.362967,5.363422,95.015370


In [24]:
delta = daily_data.groupby("Ticker")["Adj Close"].diff()

gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)

avg_gain = gain.groupby(daily_data["Ticker"]).transform(
    lambda x: x.ewm(
        alpha=1/14,
        adjust=False,
        min_periods=14
    ).mean()
)

avg_loss = loss.groupby(daily_data["Ticker"]).transform(
    lambda x: x.ewm(
        alpha=1/14,
        adjust=False,
        min_periods=14
    ).mean()
)

rs = avg_gain / avg_loss

daily_data["RSI_14"] = 100 - (100 / (1 + rs))

# Handle edge cases
daily_data.loc[
    (avg_loss == 0) & (avg_gain > 0), "RSI_14"
] = 100

daily_data.loc[
    (avg_loss == 0) & (avg_gain == 0), "RSI_14"
] = 50

In [25]:
print("Dataset Shape:", daily_data.shape)

print("\nMissing Values:")
print(daily_data["RSI_14"].isna().sum())

print("\nSummary Statistics:")
display(daily_data["RSI_14"].describe())

Dataset Shape: (63574, 65)

Missing Values:
140

Summary Statistics:


count    63434.000000
mean        53.383992
std         11.979229
min          7.385830
25%         44.983766
50%         53.561471
75%         61.739231
max         90.940117
Name: RSI_14, dtype: float64

In [26]:
# Calculate EMA 12 and EMA 26
ema_12 = daily_data.groupby("Ticker")["Adj Close"].transform(
    lambda x: x.ewm(
        span=12,
        adjust=False,
        min_periods=12
    ).mean()
)

ema_26 = daily_data.groupby("Ticker")["Adj Close"].transform(
    lambda x: x.ewm(
        span=26,
        adjust=False,
        min_periods=26
    ).mean()
)

# MACD
daily_data["MACD"] = ema_12 - ema_26

# Signal Line
daily_data["MACD_Signal"] = (
    daily_data.groupby("Ticker")["MACD"].transform(
        lambda x: x.ewm(
            span=9,
            adjust=False,
            min_periods=9
        ).mean()
    )
)

# MACD Histogram
daily_data["MACD_Histogram"] = (
    daily_data["MACD"] - daily_data["MACD_Signal"]
)

In [27]:
macd_features = [
    "MACD",
    "MACD_Signal",
    "MACD_Histogram"
]

print("Dataset Shape:", daily_data.shape)

print("\nMissing Values:")
print(daily_data[macd_features].isna().sum())

print("\nSummary Statistics:")
print(daily_data[macd_features].describe().T)

Dataset Shape: (63574, 68)

Missing Values:
MACD              250
MACD_Signal       330
MACD_Histogram    330
dtype: int64

Summary Statistics:
                  count      mean       std        min       25%       50%  \
MACD            63324.0  0.153335  1.114975 -16.912446 -0.113405  0.064864   
MACD_Signal     63244.0  0.152826  1.050730 -15.094499 -0.104152  0.061933   
MACD_Histogram  63244.0  0.000692  0.334056  -5.673820 -0.063727  0.000222   

                     75%        max  
MACD            0.331913  14.296355  
MACD_Signal     0.319144  11.995421  
MACD_Histogram  0.067709   4.865464  


# Daily Feature Engineering

Daily feature engineering was performed to extract meaningful information from historical stock prices and transform raw market data into useful inputs for machine learning models.

The features were calculated separately for each stock ticker to ensure that historical observations from different companies were not mixed. The dataset contains ten Canadian stocks listed on the Toronto Stock Exchange (TSX).

### 1. Daily Returns

The Daily_Return feature measures the percentage change in a stock's adjusted closing price compared with its previous trading observation.

**Formula:**

Daily_Return = (Adj Close(t) / Adj Close(t-1) - 1) × 100

This feature captures short-term price movements and provides a basic measure of daily stock performance.

**Statistics:**

- Mean: 0.0733%
- Standard deviation: 1.9452%
- Minimum: -29.1804%
- Maximum: 32.4627%

The average daily return was slightly positive, while the relatively high standard deviation reflects variations in daily market movements.

### 2. Return Lag Features

Four lagged return features were created:

- Return_Lag_1
- Return_Lag_5
- Return_Lag_10
- Return_Lag_20

These features represent historical daily returns from 1, 5, 10, and 20 trading observations earlier.

They allow machine learning models to identify potential relationships between previous price movements and subsequent stock performance.

### 3. Rolling Volatility

Three rolling volatility features were calculated:

- Volatility_5D
- Volatility_10D
- Volatility_20D

These features measure the standard deviation of daily returns over the previous 5, 10, and 20 observations.

**Statistics:**

| Feature        | Mean   | Standard Deviation | Minimum | Maximum |
| -------------- | ------ | ------------------ | ------- | ------- |
| Volatility_5D  | 1.5264 | 1.2237             | 0.0426  | 16.5760 |
| Volatility_10D | 1.5928 | 1.1303             | 0.1645  | 14.8759 |
| Volatility_20D | 1.6316 | 1.0685             | 0.2376  | 13.9519 |

The results indicate that longer observation windows generally produced higher average volatility, while short windows captured more immediate fluctuations.

### 4. Moving Averages and Moving Average Ratios

Four moving averages were created:

- MA_5
- MA_10
- MA_20
- MA_50

These features represent the average adjusted closing price over the previous 5, 10, 20, and 50 trading observations.

To make the features more comparable across stocks with different price levels, four additional ratio features were calculated:

- MA_Ratio_5
- MA_Ratio_10
- MA_Ratio_20
- MA_Ratio_50

These ratios measure the percentage difference between the current adjusted closing price and its corresponding moving average.

**Formula:**

MA_Ratio = (Adj Close / MA - 1) × 100

A positive value indicates that the current adjusted closing price is above its moving average, while a negative value indicates that it is below.

The mean ratios were:

- MA_Ratio_5: 0.1155%
- MA_Ratio_10: 0.2667%
- MA_Ratio_20: 0.5686%
- MA_Ratio_50: 1.4724%

These features provide information about short- and medium-term price trends.

### 5. Momentum Features

Three momentum features were created:

- Momentum_5D
- Momentum_10D
- Momentum_20D

They measure the percentage change in adjusted closing prices over the previous 5, 10, and 20 trading observations.

**Formula:**

Momentum = (Adj Close(t) / Adj Close(t-n) - 1) × 100

The average momentum values were:

- Momentum_5D: 0.3621%
- Momentum_10D: 0.7205%
- Momentum_20D: 1.4257%

The positive averages suggest that historical price changes were slightly positive on average over these observation windows.

### 6. Relative Strength Index (RSI)

The RSI_14 feature was calculated using a 14-period Exponential Moving Average-based smoothing method.

RSI measures the strength of recent upward and downward price movements and ranges from 0 to 100.

Common interpretations include:

- Above 70: Potentially overbought conditions.
- Below 30: Potentially oversold conditions.
- Around 50: Relatively balanced momentum.

**Statistics:**

- Mean: 53.3840
- Standard deviation: 11.9792
- Minimum: 7.3858
- Median: 53.5615
- Maximum: 90.9401

The average RSI was slightly above 50, indicating a modest positive momentum tendency across the observations.

### 7. MACD Features

The Moving Average Convergence Divergence (MACD) indicator was calculated using exponential moving averages.

Three features were created:

- MACD: Difference between the 12-period and 26-period exponential moving averages.
- MACD_Signal: 9-period exponential moving average of MACD.
- MACD_Histogram: Difference between MACD and its signal line.

These features capture changes in price momentum and potential trend shifts.

**Statistics:**

| Feature        | Mean   | Standard Deviation | Minimum   | Maximum |
| -------------- | ------ | ------------------ | --------- | ------- |
| MACD           | 0.1533 | 1.1150             | \-16.9124 | 14.2964 |
| MACD_Signal    | 0.1528 | 1.0507             | \-15.0945 | 11.9954 |
| MACD_Histogram | 0.0007 | 0.3341             | \-5.6738  | 4.8655  |

The histogram's mean was close to zero, indicating that MACD and its signal line were relatively balanced on average.

### 8. Calendar Features

Seven calendar-related features were created to capture recurring temporal patterns:

- DayOfWeek
- Month
- Quarter
- DayOfWeek_Sin
- DayOfWeek_Cos
- Month_Sin
- Month_Cos

The sine and cosine transformations encode weekday and monthly cycles as continuous numerical values.

This representation helps models capture cyclical patterns without treating December and January, for example, as distant categories.

The weekday cycle uses five trading days, while the monthly cycle uses twelve months.

### Missing Values

Some missing values were generated naturally during feature engineering:

- Lag features require previous observations.
- Rolling volatility and moving averages require sufficient historical windows.
- RSI and MACD require initial observations to initialize their calculations.
- Daily returns require a previous closing price.

These missing values were expected and were not automatically replaced with zeros.

### Conclusion

The daily feature engineering process generated technical, historical, momentum, volatility, and calendar features from the original stock prices.

Together, these features provide different perspectives on historical market behavior and establish a richer input dataset for subsequent machine learning experiments.

# Daily Feature Engineering – Summary

Daily feature engineering was performed to extract useful information from historical stock prices and create additional variables for machine learning.

The following feature groups were generated:

| Feature Group         | Features Created                            | Purpose                                          |
| --------------------- | ------------------------------------------- | ------------------------------------------------ |
| Daily Returns         | Daily_Return                                | Measure daily stock performance.                 |
| Return Lags           | Return_Lag_1, 5, 10, 20                     | Capture previous price movements.                |
| Rolling Volatility    | Volatility_5D, 10D, 20D                     | Measure historical price fluctuations.           |
| Moving Averages       | MA_5, 10, 20, 50                            | Identify historical price trends.                |
| Moving Average Ratios | MA_Ratio_5, 10, 20, 50                      | Measure price distance from moving averages.     |
| Momentum              | Momentum_5D, 10D, 20D                       | Measure recent price momentum.                   |
| RSI                   | RSI_14                                      | Evaluate the strength of recent price movements. |
| MACD                  | MACD, MACD_Signal, MACD_Histogram           | Capture momentum and trend changes.              |
| Calendar              | DayOfWeek, Month, Quarter, Sin/Cos features | Capture recurring calendar patterns.             |

The generated features provide information about daily performance, historical trends, volatility, momentum, and seasonal behavior.

Missing values were mainly caused by the historical lookback requirements of lagged and rolling calculations. They were retained without arbitrary imputation.

**Final Outcome:** Multiple daily features were successfully engineered, expanding the dataset with historical price indicators and calendar information for the next machine learning stage.

# Daily Economic Features

In [28]:
daily_data["Yield_Spread"] = (
    daily_data["Bond_Yield_10Y"] -
    daily_data["Policy_Rate"]
)

In [29]:
# 1. Calculate Policy Rate Change
policy_daily = (
    daily_data[["Date", "Policy_Rate"]]
    .drop_duplicates(subset="Date")
    .sort_values("Date")
    .copy()
)

policy_daily["Policy_Rate_Change_21D"] = (
    policy_daily["Policy_Rate"].diff(21)
)

In [30]:
daily_data["USD_CAD_Change_21D"] = (
    daily_data.groupby("Ticker")["USD_CAD"]
    .transform(
        lambda x: x.pct_change(
            periods=21,
            fill_method=None
        ) * 100
    )
)

In [31]:
economic_features = [
    "Yield_Spread",
    "Policy_Rate_Change_21D",
    "USD_CAD_Change_21D"
]

print("Dataset Shape:", daily_data.shape)

print("\nMissing Values:")
print(daily_data[economic_features].isna().sum())

print("\nSummary Statistics:")
print(daily_data[economic_features].describe().T)

Dataset Shape: (63574, 71)

Missing Values:
Yield_Spread              21294
Policy_Rate_Change_21D    21320
USD_CAD_Change_21D        39324
dtype: int64

Summary Statistics:
                          count      mean       std       min       25%  \
Yield_Spread            42280.0  0.802406  1.109566 -1.940000  0.320000   
Policy_Rate_Change_21D  42254.0  0.009810  0.168923 -1.500000  0.000000   
USD_CAD_Change_21D      24250.0  0.067589  1.825518 -5.735604 -1.142857   

                            50%       75%       max  
Yield_Spread            0.94000  1.290000  3.470000  
Policy_Rate_Change_21D  0.00000  0.000000  1.000000  
USD_CAD_Change_21D      0.05918  1.228916  9.553322  


## Daily Economic Features

Three daily economic features were engineered to capture the relationship between Canadian monetary conditions, bond-market expectations, and currency movements.

### 1. Yield Spread

The Yield_Spread feature was calculated as the difference between the 10-year Canadian government bond yield and the Bank of Canada's policy rate.

**Formula:**

Yield_Spread = Bond_Yield_10Y - Policy_Rate

This feature represents the difference between long-term and short-term interest rates and provides information about the yield curve. A higher spread indicates a larger difference between long-term and policy rates, while a negative spread indicates that the 10-year yield is below the policy rate.

**Statistics:**

- Mean: 0.8024
- Standard deviation: 1.1096
- Minimum: -1.94
- Median: 0.94
- Maximum: 3.47

The average positive spread indicates that long-term bond yields were generally higher than the policy rate over the available observations.

### 2. Policy Rate Change (21 Days)

The Policy_Rate_Change_21D feature measures the change in the Bank of Canada's policy rate over 21 observations.

**Formula:**

Policy_Rate_Change_21D = Policy_Rate(t) - Policy_Rate(t-21)

This feature captures recent monetary policy changes and helps represent the direction and magnitude of interest-rate adjustments.

**Statistics:**

- Mean: 0.0098
- Standard deviation: 0.1689
- Minimum: -1.50
- Median: 0.00
- Maximum: 1.00

The median of zero indicates that the policy rate usually remained unchanged over the selected observation window.

**Methodological Note:** The calculation currently uses 21 stock trading observations rather than exactly 21 calendar days. Since Bank of Canada observations are merged into the stock dataset, this definition should be documented or refined using the original unique daily economic series.

### 3. USD/CAD Change (21 Days)

The USD_CAD_Change_21D feature measures the percentage change in the USD/CAD exchange rate over 21 observations.

**Formula:**

USD_CAD_Change_21D = ((USD_CAD(t) / USD_CAD(t-21)) - 1) × 100

This feature captures recent currency movements. An increase in USD/CAD generally indicates that the US dollar has strengthened relative to the Canadian dollar, while a decrease indicates relative Canadian dollar appreciation.

**Statistics:**

- Mean: 0.0676%
- Standard deviation: 1.8255%
- Minimum: -5.7356%
- Median: 0.0592%
- Maximum: 9.5533%

The results indicate that exchange-rate movements varied considerably over the observation period.

### Missing Values Analysis

Missing values were observed in the economic features:

- Yield_Spread: 21,294 missing values.
- Policy_Rate_Change_21D: 21,320 missing values.
- USD_CAD_Change_21D: 39,324 missing values.

These missing values are mainly related to differences in historical data coverage, unavailable source observations, and the initial periods required to calculate 21-observation changes.

No arbitrary values were inserted to replace these missing observations.

**Conclusion:**

The three features provide additional economic context for the machine learning dataset by representing interest-rate differences, monetary policy changes, and currency movements. However, their historical coverage and calculation windows must be considered during feature selection and model evaluation.

## Daily Economic Features – Summary

Three economic features were created using Bank of Canada data:

1. **Yield_Spread:** Measures the difference between the 10-year bond yield and the policy rate. Its mean was 0.8024 percentage points, indicating that long-term yields were generally higher.
2. **Policy_Rate_Change_21D:** Measures the policy-rate change over 21 observations. Its mean was 0.0098, while the median was zero, suggesting that the policy rate frequently remained unchanged.
3. **USD_CAD_Change_21D:** Measures the percentage change in the USD/CAD exchange rate over 21 observations. Its mean was 0.0676%, with a standard deviation of 1.8255%.

Missing values were retained without arbitrary imputation because they mainly reflect differences in historical coverage and the calculation requirements of rolling changes.

**Final Outcome:** Three economic features were added to capture monetary policy, bond-market conditions, and exchange-rate movements.

# Target Variable Creation

In [32]:
daily_data = daily_data.sort_values(
    ["Ticker", "Date"]
).reset_index(drop=True)

In [33]:
daily_data["Future_Return_5D"] = (
    daily_data.groupby("Ticker")["Adj Close"]
    .transform(
        lambda x: (x.shift(-5) / x - 1) * 100
    )
)

In [34]:
print("Dataset Shape:", daily_data.shape)

print("\nMissing Values:")
print(daily_data["Future_Return_5D"].isna().sum())

print("\nSummary Statistics:")
print(daily_data["Future_Return_5D"].describe())

print("\nSample:")
display(
    daily_data[
        ["Date", "Ticker", "Adj Close", "Future_Return_5D"]
    ].tail(15)
)

Dataset Shape: (63574, 72)

Missing Values:
50

Summary Statistics:
count    63524.000000
mean         0.362051
std          4.238297
min        -47.013487
25%         -1.666637
50%          0.373576
75%          2.314759
max         43.282065
Name: Future_Return_5D, dtype: float64

Sample:


,Date,Ticker,Adj Close,Future_Return_5D
63559,2026-10-19,TD.TO,169.610001,3.625962
63560,2026-10-20,TD.TO,171.710007,2.830354
63561,2026-10-21,TD.TO,175.110001,1.016503
63562,2026-10-22,TD.TO,175.139999,1.707209
63563,2026-10-23,TD.TO,174.470001,-0.246465
63564,2026-10-26,TD.TO,175.759995,-0.432405
63565,2026-10-27,TD.TO,176.570007,-1.030757
63566,2026-10-28,TD.TO,176.889999,1.153255
63567,2026-10-29,TD.TO,178.130005,-3.132545
63568,2026-10-30,TD.TO,174.039993,-0.643527


In [35]:
output_path = DATA_DIR / "canada_integrated_daily_macro_features1.csv"

daily_data.to_csv(output_path, index=False)

print("Saved successfully:", output_path)

Saved successfully: data/processed/main/canada_integrated_daily_macro_features1.csv


## Target Variable Creation

A target variable named Future_Return_5D was created to represent the percentage return of each stock over the next five trading observations. The objective is to provide the machine learning models with a measurable future outcome that can be predicted using historical stock and economic information.

The target was calculated separately for each ticker using the adjusted closing price (Adj Close).

**Formula:**

Future_Return_5D = ((Adj Close(t+5) / Adj Close(t)) - 1) × 100

The calculation uses groupby("Ticker") to ensure that future prices are taken from the same stock rather than from another ticker.

Adjusted closing prices were selected because they account for historical dividends and stock splits, making them suitable for measuring historical total-return performance.

### Target Statistics

| Statistic                 | Value      |
| ------------------------- | ---------- |
| Total observations        | 63,574     |
| Valid target observations | 63,524     |
| Missing target values     | 50         |
| Mean                      | 0.3621%    |
| Standard deviation        | 4.2383%    |
| Minimum                   | \-47.0135% |
| 25th Percentile           | \-1.6666%  |
| Median                    | 0.3736%    |
| 75th Percentile           | 2.3148%    |
| Maximum                   | 43.2821%   |

### Missing Values

The target contains 50 missing values, corresponding to the final five observations of each of the ten stocks.

These values are expected because there are no subsequent five trading observations available to calculate future returns.

They should not be filled with zero because doing so would incorrectly imply that the stock experienced no price change.

### Mathematical Validation

To verify the target calculation, the future date and future adjusted closing price were independently generated using a separate five-observation shift.

The resulting forward returns were compared with the original Future_Return_5D values.

The maximum absolute difference between both calculations was **0.0**, confirming that the target was calculated correctly.

### Extreme Target Values

The target includes substantial positive and negative returns, particularly during periods of market volatility.

For example:

- The highest observed five-day return was approximately +43.28% for SU.TO in November 2008.
- The lowest observed five-day return was approximately -47.01% for CNQ.TO in March 2020.

These observations were investigated as part of the outlier analysis. They were retained because there was not sufficient evidence to classify them as data errors.

### Important Modeling Considerations

Future_Return_5D is the target variable and must not be included among the model input features.

The final five observations per ticker should be excluded from supervised model training because their target values are unavailable.

Additionally, chronological data splitting and an appropriate five-observation purge or embargo should be considered to reduce information leakage between training, validation, and test periods.

**Conclusion:**

The Future_Return_5D target was successfully created and mathematically validated. It provides a consistent measure of five-observation forward stock returns and establishes the prediction objective for the subsequent machine learning stage.

## Target Creation – Summary

A target variable named Future_Return_5D was created to measure each stock's percentage return over the next five trading observations.

The target was calculated using adjusted closing prices and grouped by ticker to prevent mixing observations between different stocks.

**Key Results:**

- Total observations: 63,574.
- Valid target values: 63,524.
- Missing values: 50, corresponding to the final five observations of each stock.
- Mean return: 0.3621%.
- Standard deviation: 4.2383%.
- Minimum return: -47.0135%.
- Maximum return: +43.2821%.

The calculation was independently validated, producing a maximum absolute difference of **0.0**.

The missing target values were retained because future observations are unavailable. Extreme returns were also retained following the initial outlier investigation.

**Final Outcome:** The five-observation forward return target was successfully created and validated for subsequent machine learning modeling.

# Final Feature Engineering Audit

In [4]:
DATA_DIR = "data/processed/main/"
daily_data = pd.read_csv(
    DATA_DIR + "canada_integrated_daily_macro_features1.csv",
    parse_dates=[
        "Date",
        "Bank_Availability_Date",
        "Macro_Reference_Date",
        "Macro_Availability_Date"
    ]
)

daily_data = daily_data.sort_values(
    ["Ticker", "Date"]
).reset_index(drop=True)

print("Shape:", daily_data.shape)
print("Tickers:", daily_data["Ticker"].nunique())
print("Date Range:", daily_data["Date"].min(), "to", daily_data["Date"].max())

daily_data.head()

Shape: (63574, 72)
Tickers: 10
Date Range: 2000-01-04 00:00:00 to 2026-11-06 00:00:00


,Date,Ticker,Open,High,Low,Close,Adj Close,Volume,Dividends,Stock Splits,...,Momentum_10D,Momentum_20D,RSI_14,MACD,MACD_Signal,MACD_Histogram,Yield_Spread,Policy_Rate_Change_21D,USD_CAD_Change_21D,Future_Return_5D
0,2000-01-04,ABX.TO,25.150000,25.65,24.950001,25.450001,17.804482,926500,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.589415
1,2000-01-05,ABX.TO,25.400000,26.00,25.350000,25.650000,17.944401,1452200,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.364461
2,2000-01-06,ABX.TO,25.799999,26.00,25.500000,25.799999,18.049345,600200,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-3.100806
3,2000-01-07,ABX.TO,25.850000,26.00,25.100000,25.250000,17.664574,1020800,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-2.376280
4,2000-01-10,ABX.TO,25.350000,25.35,25.150000,25.250000,17.664574,1281100,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.188113


In [5]:
missing_report = pd.DataFrame({
    "Missing_Count": daily_data.isna().sum(),
    "Missing_Percentage": (
        daily_data.isna().mean() * 100
    )
})

missing_report = missing_report[
    missing_report["Missing_Count"] > 0
].sort_values(
    "Missing_Percentage",
    ascending=False
)

print(missing_report)

                        Missing_Count  Missing_Percentage
USD_CAD_Change_21D              39324           61.855476
USD_CAD                         38984           61.320666
Policy_Rate_Change_21D          21320           33.535722
Yield_Spread                    21294           33.494825
Policy_Rate                     21090           33.173939
Bond_Yield_10Y                   2473            3.889955
Bank_Availability_Date           2259            3.553339
MA_50                             490            0.770755
MA_Ratio_50                       490            0.770755
MACD_Histogram                    330            0.519080
MACD_Signal                       330            0.519080
MACD                              250            0.393243
Return_Lag_20                     210            0.330324
Volatility_20D                    200            0.314594
Momentum_20D                      200            0.314594
MA_20                             190            0.298864
MA_Ratio_20   

In [6]:
print("Duplicate Rows:",
      daily_data.duplicated().sum())

print(
    "Duplicate Date-Ticker:",
    daily_data.duplicated(
        subset=["Date", "Ticker"]
    ).sum()
)

Duplicate Rows: 0
Duplicate Date-Ticker: 0


In [7]:
numeric_data = daily_data.select_dtypes(
    include=np.number
)

infinite_counts = pd.Series(
    np.isinf(numeric_data).sum()
)

infinite_counts = infinite_counts[
    infinite_counts > 0
]

print("Infinite Values:")
display(infinite_counts)

Infinite Values:


Series([], dtype: int64)

In [8]:
print("Data Types:")
print(
    daily_data.dtypes.to_frame("Data_Type")
)

Data Types:
                             Data_Type
Date                    datetime64[ns]
Ticker                          object
Open                           float64
High                           float64
Low                            float64
...                                ...
MACD_Histogram                 float64
Yield_Spread                   float64
Policy_Rate_Change_21D         float64
USD_CAD_Change_21D             float64
Future_Return_5D               float64

[72 rows x 1 columns]


In [9]:
target_outliers = daily_data[
    ["Date", "Ticker", "Adj Close", "Future_Return_5D"]
].dropna(
    subset=["Future_Return_5D"]
)

print("Top 10 Highest Future Returns:")
display(
    target_outliers.nlargest(
        10, "Future_Return_5D"
    )
)

print("Top 10 Lowest Future Returns:")
display(
    target_outliers.nsmallest(
        10, "Future_Return_5D"
    )
)

Top 10 Highest Future Returns:


,Date,Ticker,Adj Close,Future_Return_5D
52323,2008-11-20,SU.TO,10.975168,43.282065
48436,2020-04-13,SHOP.TO,62.050999,43.037177
48435,2020-04-09,SHOP.TO,58.472000,42.208237
22469,2008-11-20,CNQ.TO,4.744168,42.026482
49583,2024-11-06,SHOP.TO,114.220001,41.131146
25316,2020-03-27,CNQ.TO,4.695951,40.991019
25313,2020-03-24,CNQ.TO,4.826395,40.613548
55170,2020-03-27,SU.TO,12.623557,39.866096
47285,2015-09-10,SHOP.TO,3.400000,38.970582
52324,2008-11-21,SU.TO,12.018486,38.700265


Top 10 Lowest Future Returns:


,Date,Ticker,Adj Close,Future_Return_5D
25304,2020-03-11,CNQ.TO,7.036139,-47.013487
25300,2020-03-05,CNQ.TO,11.042264,-46.408840
55158,2020-03-11,SU.TO,19.899584,-41.814667
25303,2020-03-10,CNQ.TO,7.476744,-41.251144
55154,2020-03-05,SU.TO,26.729979,-40.931289
25299,2020-03-04,CNQ.TO,11.225290,-37.318864
25297,2020-03-02,CNQ.TO,11.570994,-37.229086
25306,2020-03-13,CNQ.TO,6.727715,-36.331002
55157,2020-03-10,SU.TO,21.313295,-35.472233
25301,2020-03-06,CNQ.TO,10.255951,-34.401841


#  Outlier Investigation

An outlier investigation was conducted to examine unusually large daily and five-day forward stock returns in the Canadian stock market dataset. The main objective was to distinguish genuine market movements from potential data errors or corporate-action effects before proceeding with model development.
The investigation focused on extreme values in Daily_Return and Future_Return_5D, particularly during periods of significant market volatility, such as the 2008 financial crisis and the COVID-19 market disruption in 2020.
To validate the extreme five-day returns, the future date and adjusted closing price were independently recalculated using the complete chronological dataset for each ticker. The calculated returns were then compared with the original Future_Return_5D values. The maximum absolute difference was 0.0, confirming that the target calculations were mathematically consistent.
The investigation also examined daily OHLC prices and corporate actions within the relevant forward-return windows. Many extreme movements were supported by corresponding changes in the observed daily prices, particularly during the 2008 and 2020 market disruptions. No stock splits were identified in the inspected windows that could explain the major movements. Some dividend events were observed, but they were not sufficient on their own to explain the large return variations.
However, several unusually large daily movements in Shopify (SHOP.TO) require additional verification. Examples include:
    • May 4, 2023: approximately +23.16%.
    • November 2, 2023: approximately +21.31%.
    • November 12, 2024: approximately +21.45%.
    • August 5, 2026: approximately +16.47%.
These movements may be associated with company-specific events or market developments, but their causes have not yet been independently confirmed.
Conclusion:
The investigation found no sufficient evidence to justify automatically removing or clipping extreme returns. Many observed movements appear consistent with the corresponding historical price changes. Therefore, the outliers were retained to preserve potentially meaningful market information. The unusually large Shopify movements remain flagged for further verification before final model development.

In [10]:
daily_data_copy = daily_data.copy(deep=True)

In [11]:
# Sort the original dataset
daily_data = daily_data.sort_values(
    ["Ticker", "Date"]
).reset_index(drop=True)

# Calculate future date and adjusted close on full data
daily_data["Future_Date_Check"] = (
    daily_data.groupby("Ticker")["Date"].shift(-5)
)

daily_data["Future_Adj_Close_Check"] = (
    daily_data.groupby("Ticker")["Adj Close"].shift(-5)
)

# Select extreme returns after calculating future values
extreme_returns = daily_data[
    daily_data["Future_Return_5D"].abs() > 20
].copy()

# Display verification results
display(
    extreme_returns[
        [
            "Date",
            "Ticker",
            "Adj Close",
            "Future_Date_Check",
            "Future_Adj_Close_Check",
            "Future_Return_5D",
            "Dividends",
            "Stock Splits"
        ]
    ].sort_values(
        "Future_Return_5D",
        ascending=False
    ).head(20)
)

,Date,Ticker,Adj Close,Future_Date_Check,Future_Adj_Close_Check,Future_Return_5D,Dividends,Stock Splits
52323,2008-11-20,SU.TO,10.975168,2008-11-27,15.725448,43.282065,0.0,0.0
48436,2020-04-13,SHOP.TO,62.050999,2020-04-20,88.755997,43.037177,0.0,0.0
48435,2020-04-09,SHOP.TO,58.472000,2020-04-17,83.152000,42.208237,0.0,0.0
22469,2008-11-20,CNQ.TO,4.744168,2008-11-27,6.737975,42.026482,0.0,0.0
49583,2024-11-06,SHOP.TO,114.220001,2024-11-13,161.199997,41.131146,0.0,0.0
25316,2020-03-27,CNQ.TO,4.695951,2020-04-03,6.620869,40.991019,0.0,0.0
25313,2020-03-24,CNQ.TO,4.826395,2020-03-31,6.786565,40.613548,0.0,0.0
55170,2020-03-27,SU.TO,12.623557,2020-04-03,17.656076,39.866096,0.0,0.0
47285,2015-09-10,SHOP.TO,3.400000,2015-09-17,4.725000,38.970582,0.0,0.0
52324,2008-11-21,SU.TO,12.018486,2008-11-28,16.669672,38.700265,0.0,0.0


In [12]:
daily_data["Future_Return_Check"] = (
    (
        daily_data["Future_Adj_Close_Check"] /
        daily_data["Adj Close"] - 1
    ) * 100
)

difference = (
    daily_data["Future_Return_5D"] -
    daily_data["Future_Return_Check"]
).abs()

print(
    "Maximum Difference:",
    difference.max()
)

Maximum Difference: 7.105427357601002e-15


In [13]:
print("Total Dividend Events:",
      (daily_data["Dividends"] != 0).sum())

print("Total Stock Split Events:",
      (daily_data["Stock Splits"] != 0).sum())

print("\nStock Splits:")
display(
    daily_data.loc[
        daily_data["Stock Splits"] != 0,
        ["Date", "Ticker", "Close", "Adj Close", "Stock Splits"]
    ]
)

print("\nDividend Statistics:")
display(
    daily_data.loc[
        daily_data["Dividends"] != 0,
        ["Dividends"]
    ].describe()
)

Total Dividend Events: 934
Total Stock Split Events: 26

Stock Splits:


,Date,Ticker,Close,Adj Close,Stock Splits
7037,2001-03-01,BMO.TO,37.700001,12.951309,2.000
14591,2004-05-19,BN.TO,3.529671,2.564955,1.500
15071,2006-04-17,BN.TO,7.181979,5.366194,1.500
15347,2007-05-22,BN.TO,10.916732,8.268441,1.500
16827,2013-04-15,BN.TO,8.831402,7.558238,1.033
17349,2015-05-13,BN.TO,15.564538,13.731295,1.500
18576,2020-04-02,BN.TO,22.225815,20.969034,1.500
19252,2022-12-12,BN.TO,30.726667,29.924658,1.237
19962,2025-10-10,BN.TO,60.290001,59.898788,1.500
21335,2004-05-19,CNQ.TO,4.689030,2.497998,2.000



Dividend Statistics:


,Dividends
count,934.000000
mean,0.386659
std,0.384973
min,0.006121
25%,0.081375
50%,0.240500
75%,0.613750
max,1.760000


In [14]:
# Select extreme cases
extreme_cases = daily_data[
    daily_data["Future_Return_5D"].abs() >= 30
].copy()

# Inspect daily prices during each 5-day return period
for _, row in extreme_cases.iterrows():

    ticker = row["Ticker"]
    start_date = row["Date"]
    end_date = row["Future_Date_Check"]

    window = daily_data[
        (daily_data["Ticker"] == ticker) &
        (daily_data["Date"] >= start_date) &
        (daily_data["Date"] <= end_date)
    ]

    print("\n" + "=" * 75)
    print(f"Ticker: {ticker}")
    print(f"Period: {start_date.date()} -> {end_date.date()}")
    print(f"Future Return: {row['Future_Return_5D']:.2f}%")

    print(
        window[
            [
                "Date",
                "Open",
                "High",
                "Low",
                "Close",
                "Adj Close",
                "Daily_Return",
                "Dividends",
                "Stock Splits"
            ]
        ]
    )


Ticker: ABX.TO
Period: 2008-09-15 -> 2008-09-22
Future Return: 33.50%
           Date       Open       High        Low      Close  Adj Close  \
2190 2008-09-15  31.750000  31.959999  29.299999  29.670000  21.523413   
2191 2008-09-16  28.719999  31.010000  28.440001  30.870001  22.393925   
2192 2008-09-17  30.690001  35.849998  30.340000  35.310001  25.614824   
2193 2008-09-18  36.349998  38.180000  31.540001  32.500000  23.576372   
2194 2008-09-19  32.750000  36.500000  32.750000  36.500000  26.478081   
2195 2008-09-22  37.910000  40.900002  37.910000  39.610001  28.734156   

      Daily_Return  Dividends  Stock Splits  
2190     -5.719752        0.0           0.0  
2191      4.044489        0.0           0.0  
2192     14.382917        0.0           0.0  
2193     -7.958095        0.0           0.0  
2194     12.307698        0.0           0.0  
2195      8.520538        0.0           0.0  

Ticker: ABX.TO
Period: 2008-10-09 -> 2008-10-17
Future Return: -30.65%
           Date 

# Outlier Investigation – Summary
An investigation was conducted to identify whether extreme stock returns represented genuine market movements or potential data-quality issues.
The five-day forward returns were mathematically validated, with a maximum calculation difference of 0.0. Daily OHLC prices and corporate actions were also inspected. Many extreme movements were consistent with historical market volatility, particularly during 2008 and March 2020, and no obvious stock-split explanation was found in the inspected periods.
Several unusually large Shopify (SHOP.TO) movements were identified and require independent verification.
Final Decision: Extreme values were retained without automatic removal or clipping because they may represent meaningful market behavior. Additional verification of selected Shopify price movements remains necessary.

# OHLC Integrity

In [15]:
# 1. Check positive prices
invalid_prices = daily_data[
    (daily_data["Open"] <= 0) |
    (daily_data["High"] <= 0) |
    (daily_data["Low"] <= 0) |
    (daily_data["Close"] <= 0) |
    (daily_data["Adj Close"] <= 0)
]

# 2. Check OHLC consistency
invalid_ohlc = daily_data[
    (daily_data["High"] < daily_data["Open"]) |
    (daily_data["High"] < daily_data["Close"]) |
    (daily_data["Low"] > daily_data["Open"]) |
    (daily_data["Low"] > daily_data["Close"]) |
    (daily_data["High"] < daily_data["Low"])
]

# 3. Check negative volume
invalid_volume = daily_data[
    daily_data["Volume"] < 0
]

# 4. Display results
print("Positive Price Violations:", len(invalid_prices))
print("OHLC Inconsistencies:", len(invalid_ohlc))
print("Negative Volume Records:", len(invalid_volume))

Positive Price Violations: 0
OHLC Inconsistencies: 0
Negative Volume Records: 0


In [16]:
if len(invalid_prices) > 0:
    display(invalid_prices)

if len(invalid_ohlc) > 0:
    display(invalid_ohlc)

if len(invalid_volume) > 0:
    display(invalid_volume)

In [17]:
# Sort data chronologically
daily_data = daily_data.sort_values(
    ["Ticker", "Date"]
).reset_index(drop=True)

# Daily Returns
daily_data["Close_Daily_Return"] = (
    daily_data.groupby("Ticker")["Close"]
    .pct_change(fill_method=None) * 100
)

daily_data["Adj_Close_Daily_Return"] = (
    daily_data.groupby("Ticker")["Adj Close"]
    .pct_change(fill_method=None) * 100
)

# Five-day Forward Returns
daily_data["Close_Future_Return_5D"] = (
    daily_data.groupby("Ticker")["Close"]
    .transform(lambda x: (x.shift(-5) / x - 1) * 100)
)

daily_data["Adj_Close_Future_Return_5D"] = (
    daily_data.groupby("Ticker")["Adj Close"]
    .transform(lambda x: (x.shift(-5) / x - 1) * 100)
)

In [18]:
daily_comparison = (
    daily_data[
        ["Close_Daily_Return", "Adj_Close_Daily_Return"]
    ]
    .describe()
)

display(daily_comparison)

,Close_Daily_Return,Adj_Close_Daily_Return
count,63564.000000,63564.000000
mean,0.062741,0.073344
std,1.948448,1.945237
min,-29.180437,-29.180444
25%,-0.783749,-0.768343
50%,0.067043,0.074152
75%,0.895296,0.902516
max,32.462690,32.462666


In [19]:
daily_data["Daily_Return_Difference"] = (
    daily_data["Adj_Close_Daily_Return"]
    - daily_data["Close_Daily_Return"]
).abs()

daily_data["Future_Return_Difference"] = (
    daily_data["Adj_Close_Future_Return_5D"]
    - daily_data["Close_Future_Return_5D"]
).abs()

print("Daily Return Difference:")
display(
    daily_data["Daily_Return_Difference"].describe()
)

print("Five-Day Return Difference:")
display(
    daily_data["Future_Return_Difference"].describe()
)

Daily Return Difference:


count    63564.000000
mean         0.010621
std          0.099349
min          0.000000
25%          0.000005
50%          0.000014
75%          0.000028
max          4.091966
Name: Daily_Return_Difference, dtype: float64

Five-Day Return Difference:


count    63524.000000
mean         0.053199
std          0.217937
min          0.000000
25%          0.000006
50%          0.000016
75%          0.000032
max          5.213624
Name: Future_Return_Difference, dtype: float64

In [20]:
largest_differences = daily_data.nlargest(
    10, "Future_Return_Difference"
)[[
    "Date",
    "Ticker",
    "Close",
    "Adj Close",
    "Close_Future_Return_5D",
    "Adj_Close_Future_Return_5D",
    "Future_Return_Difference",
    "Dividends",
    "Stock Splits"
]]

display(largest_differences)

,Date,Ticker,Close,Adj Close,Close_Future_Return_5D,Adj_Close_Future_Return_5D,Future_Return_Difference,Dividends,Stock Splits
25309,2020-03-18,CNQ.TO,5.386876,3.728205,29.727263,34.940886,5.213624,0.0,0.0
25308,2020-03-17,CNQ.TO,6.346719,4.392501,5.632728,9.878055,4.245327,0.0,0.0
25307,2020-03-16,CNQ.TO,7.071499,4.894116,-15.720228,-12.333079,3.387148,0.0,0.0
25305,2020-03-12,CNQ.TO,8.550441,5.917677,-35.853378,-33.275391,2.577987,0.0,0.0
25306,2020-03-13,CNQ.TO,9.720862,6.727715,-38.790936,-36.331002,2.459933,0.0,0.0
25915,2022-08-17,CNQ.TO,34.476006,27.722569,6.392837,8.692349,2.299512,0.0,0.0
25914,2022-08-16,CNQ.TO,34.162586,27.470543,6.359041,8.657831,2.298790,0.0,0.0
3367,2013-05-24,ABX.TO,19.690001,15.212458,10.614526,12.911567,2.297041,0.0,0.0
25916,2022-08-18,CNQ.TO,35.465229,28.518017,5.201637,7.475394,2.273757,0.0,0.0
3368,2013-05-27,ABX.TO,20.059999,15.498310,9.471584,11.744912,2.273329,0.0,0.0


In [21]:
# Ensure chronological order
daily_data = daily_data.sort_values(
    ["Ticker", "Date"]
).reset_index(drop=True)

# Calculate the future date for each stock
daily_data["Future_Date_Audit"] = (
    daily_data.groupby("Ticker")["Date"].shift(-5)
)

# Select the 10 largest differences
top_differences = daily_data.nlargest(
    10, "Future_Return_Difference"
)

# Inspect corporate actions within each forward window
action_results = []

for _, row in top_differences.iterrows():

    actions = daily_data[
        (daily_data["Ticker"] == row["Ticker"]) &
        (daily_data["Date"] > row["Date"]) &
        (daily_data["Date"] <= row["Future_Date_Audit"]) &
        (
            (daily_data["Dividends"] != 0) |
            (daily_data["Stock Splits"] != 0)
        )
    ]

    for _, action in actions.iterrows():
        action_results.append({
            "Ticker": row["Ticker"],
            "Start_Date": row["Date"],
            "Future_Date": row["Future_Date_Audit"],
            "Action_Date": action["Date"],
            "Dividends": action["Dividends"],
            "Stock_Splits": action["Stock Splits"],
            "Future_Return_Difference": row["Future_Return_Difference"]
        })

actions_df = pd.DataFrame(action_results)

if not actions_df.empty:
    display(actions_df)
else:
    print("No corporate actions found in the inspected forward windows.")

,Ticker,Start_Date,Future_Date,Action_Date,Dividends,Stock_Splits,Future_Return_Difference
0,CNQ.TO,2020-03-18,2020-03-25,2020-03-19,0.208129,0.000,5.213624
1,CNQ.TO,2020-03-17,2020-03-24,2020-03-19,0.208129,0.000,4.245327
2,CNQ.TO,2020-03-16,2020-03-23,2020-03-19,0.208129,0.000,3.387148
3,CNQ.TO,2020-03-12,2020-03-19,2020-03-19,0.208129,0.000,2.577987
4,CNQ.TO,2020-03-13,2020-03-20,2020-03-19,0.208129,0.000,2.459933
5,CNQ.TO,2022-08-17,2022-08-24,2022-08-22,0.750000,1.021,2.299512
6,CNQ.TO,2022-08-16,2022-08-23,2022-08-22,0.750000,1.021,2.298790
7,ABX.TO,2013-05-24,2013-05-31,2013-05-29,0.208000,0.000,2.297041
8,ABX.TO,2013-05-24,2013-05-31,2013-05-30,0.205000,0.000,2.297041
9,CNQ.TO,2022-08-18,2022-08-25,2022-08-22,0.750000,1.021,2.273757


In [22]:
# Identify future-related and validation columns

future_columns = [
    col for col in daily_data.columns
    if any(keyword in col.lower() for keyword in [
        "future",
        "check",
        "audit"
    ])
]

print("Future / Validation Columns:")
print(future_columns)

Future / Validation Columns:
['Future_Return_5D', 'Future_Date_Check', 'Future_Adj_Close_Check', 'Future_Return_Check', 'Close_Future_Return_5D', 'Adj_Close_Future_Return_5D', 'Future_Return_Difference', 'Future_Date_Audit']


In [23]:
# Display all dataset columns

for i, col in enumerate(daily_data.columns, 1):
    print(i, col)

1 Date
2 Ticker
3 Open
4 High
5 Low
6 Close
7 Adj Close
8 Volume
9 Dividends
10 Stock Splits
11 Bank_Availability_Date
12 Policy_Rate
13 Bond_Yield_10Y
14 USD_CAD
15 Macro_Reference_Date
16 CPI
17 Inflation_YoY
18 GDP_Real_Millions_CAD
19 GDP_MoM
20 GDP_YoY
21 Employment
22 Labour_Force
23 Participation_Rate
24 Unemployment_Rate
25 BCPI_Total
26 BCPI_NonEnergy
27 BCPI_Energy
28 BCPI_Metals
29 BCPI_Forestry
30 BCPI_Agriculture
31 BCPI_Fish
32 Macro_Availability_Date
33 DayOfWeek
34 Month
35 Quarter
36 DayOfWeek_Sin
37 DayOfWeek_Cos
38 Month_Sin
39 Month_Cos
40 Macro_Feature_Availability_Date
41 BCPI_Change_12M
42 BCPI_Energy_Change_12M
43 BCPI_Metals_Change_12M
44 Unemployment_Change_12M
45 Employment_Growth_12M
46 Daily_Return
47 Return_Lag_1
48 Return_Lag_5
49 Return_Lag_10
50 Return_Lag_20
51 Volatility_5D
52 Volatility_10D
53 Volatility_20D
54 MA_5
55 MA_10
56 MA_20
57 MA_50
58 MA_Ratio_5
59 MA_Ratio_10
60 MA_Ratio_20
61 MA_Ratio_50
62 Momentum_5D
63 Momentum_10D
64 Momentum_20D
65 

In [24]:
# Temporary validation and comparison columns
columns_to_drop = [
    "Future_Date_Check",
    "Future_Adj_Close_Check",
    "Future_Return_Check",
    "Close_Daily_Return",
    "Adj_Close_Daily_Return",
    "Close_Future_Return_5D",
    "Adj_Close_Future_Return_5D",
    "Daily_Return_Difference",
    "Future_Return_Difference",
    "Future_Date_Audit"
]

# Remove temporary columns
daily_data = daily_data.drop(
    columns=columns_to_drop,
    errors="ignore"
)

print("New Dataset Shape:", daily_data.shape)

New Dataset Shape: (63574, 72)


In [25]:
print("Dataset Shape:", daily_data.shape)

print("\nRemaining Future / Validation Columns:")

print([
    col for col in daily_data.columns
    if any(keyword in col.lower()
           for keyword in ["future", "check", "audit"])
])

print("\nDuplicate Rows:",
      daily_data.duplicated().sum())

print("\nDuplicate Date-Ticker:",
      daily_data.duplicated(
          subset=["Date", "Ticker"]
      ).sum())

Dataset Shape: (63574, 72)

Remaining Future / Validation Columns:
['Future_Return_5D']

Duplicate Rows: 0

Duplicate Date-Ticker: 0


In [ ]:
DATA_DIR = "data/processed"
output_path = DATA_DIR + "canada_integrated_daily_macro_features_demo.csv"

daily_data.to_csv(output_path, index=False)

print("Saved successfully:", output_path)

# OHLC Integrity Audit

## 1. Objective

The OHLC Integrity Audit was conducted to assess the consistency and validity of historical stock price data collected for the ten selected Canadian stocks listed on the Toronto Stock Exchange (TSX).

The purpose was to identify invalid price records, inconsistent Open-High-Low-Close relationships, and invalid trading volume values before proceeding with feature engineering and model development.

## 2. Audit Scope

The audit examined the following price and volume fields:

- Open
- High
- Low
- Close
- Volume

The checks were performed on the cleaned daily stock dataset, which contains 63,574 observations covering ten Canadian stocks.

## 3. Validation Checks

Three main integrity checks were performed.

### A. Positive Price Validation

All OHLC price values were checked to ensure that they were strictly positive.

This validation helps identify invalid records containing zero or negative prices that could affect return calculations, technical indicators, and other price-derived features.

**Result: 0 positive price violations.**

### B. OHLC Consistency Validation

The relationships between the four daily price fields were examined using the following conditions:

- High must be greater than or equal to Open.
- High must be greater than or equal to Close.
- Low must be less than or equal to Open.
- Low must be less than or equal to Close.
- High must be greater than or equal to Low.

These conditions ensure that the daily trading range is internally consistent.

**Result: 0 OHLC inconsistencies.**

### C. Trading Volume Validation

Trading volume values were checked to ensure that no negative volume records existed.

Zero-volume observations were not automatically classified as invalid because they may represent special market conditions or data-provider reporting behavior.

**Result: 0 negative volume records.**

## 4. Audit Results

| Validation Check           | Violations | Status |
| -------------------------- | ---------- | ------ |
| Positive Price Validation  | 0          | Passed |
| OHLC Consistency           | 0          | Passed |
| Negative Volume Validation | 0          | Passed |

All three validation checks passed successfully.

## 5. Interpretation

The results indicate that the cleaned stock dataset contains internally consistent OHLC price records.

No zero or negative prices were detected, and all daily price relationships satisfied the expected OHLC constraints.

Similarly, no negative trading volume values were found.

These results support the reliability of the dataset for subsequent calculations involving daily returns, moving averages, momentum indicators, volatility, and other technical features.

It is important to distinguish between OHLC integrity and broader financial data accuracy. Passing these checks confirms internal consistency, but does not independently prove that every historical price matches an official exchange record.

Corporate actions, historical adjustments, and unusual price movements should therefore be considered separately when interpreting extreme returns.

## 6. Conclusion

The OHLC Integrity Audit passed all validation checks without detecting any violations.

The dataset contains valid positive price values, consistent daily OHLC relationships, and no negative trading volume records.

No additional row removal or price correction was required based on these three checks.

The cleaned stock dataset was therefore considered suitable for proceeding with the next stages of feature engineering and machine learning preparation, subject to the separate review of corporate actions and historical outliers.

# OHLC Integrity Audit – Summary

An OHLC integrity audit was performed on the cleaned Canadian stock dataset containing 63,574 daily observations across ten TSX-listed stocks.

The audit examined three main aspects:

- **Positive Prices:** No zero or negative OHLC price values were detected.
- **OHLC Consistency:** All Open, High, Low, and Close relationships satisfied the expected validation rules.
- **Trading Volume:** No negative volume records were identified.

All validation checks passed successfully:

| Check                     | Violations |
| ------------------------- | ---------- |
| Positive Price Violations | 0          |
| OHLC Inconsistencies      | 0          |
| Negative Volume Records   | 0          |

The results confirmed that the dataset was internally consistent, and no additional corrections or row removals were required based on these checks.

The dataset was considered suitable for subsequent feature engineering and model preparation.

# Temporal Leakage Audit

In [28]:
# Ensure date columns use datetime
date_columns = [
    "Date",
    "Bank_Availability_Date",
    "Macro_Reference_Date",
    "Macro_Availability_Date",
    "Macro_Feature_Availability_Date"
]

for col in date_columns:
    daily_data[col] = pd.to_datetime(daily_data[col], errors="coerce")


# 1. Bank data availability
bank_future = (
    daily_data["Bank_Availability_Date"].notna() &
    (daily_data["Bank_Availability_Date"] > daily_data["Date"])
)

# 2. Monthly macro availability
macro_future = (
    daily_data["Macro_Availability_Date"].notna() &
    (daily_data["Macro_Availability_Date"] > daily_data["Date"])
)

# 3. Macro feature availability
macro_feature_future = (
    daily_data["Macro_Feature_Availability_Date"].notna() &
    (daily_data["Macro_Feature_Availability_Date"] > daily_data["Date"])
)

# 4. Monthly reference date
macro_reference_future = (
    daily_data["Macro_Reference_Date"].notna() &
    (daily_data["Macro_Reference_Date"] > daily_data["Date"])
)

print("TEMPORAL AVAILABILITY AUDIT")
print("=" * 45)

print("Bank availability violations:", bank_future.sum())
print("Macro availability violations:", macro_future.sum())
print("Macro feature availability violations:", macro_feature_future.sum())
print("Macro reference date violations:", macro_reference_future.sum())


# Bank data age
bank_age = (
    daily_data["Date"] -
    daily_data["Bank_Availability_Date"]
).dt.days

print("\nBank Data Age:")
print(bank_age.describe())

print("\nBank records older than 7 days:", (bank_age > 7).sum())


# Check monthly availability dates consistency
print(
    "\nMacro availability date mismatches:",
    (
        daily_data["Macro_Availability_Date"] !=
        daily_data["Macro_Feature_Availability_Date"]
    ).sum()
)

TEMPORAL AVAILABILITY AUDIT
Bank availability violations: 0
Macro availability violations: 0
Macro feature availability violations: 0
Macro reference date violations: 0

Bank Data Age:
count    61315.000000
mean         0.002675
std          0.078679
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          3.000000
dtype: float64

Bank records older than 7 days: 0

Macro availability date mismatches: 0


#### Recalculating technical indicators using the Close

In [29]:
# Sort data chronologically per ticker
daily_data = daily_data.sort_values(
    ["Ticker", "Date"]
).reset_index(drop=True)

# Daily return based on Close
daily_data["Close_Daily_Return"] = (
    daily_data.groupby("Ticker")["Close"]
    .pct_change(fill_method=None) * 100
)

# Return lags
for lag in [1, 5, 10, 20]:
    daily_data[f"Close_Return_Lag_{lag}"] = (
        daily_data.groupby("Ticker")["Close_Daily_Return"]
        .shift(lag)
    )

# Rolling volatility
for window in [5, 10, 20]:
    daily_data[f"Close_Volatility_{window}D"] = (
        daily_data.groupby("Ticker")["Close_Daily_Return"]
        .transform(
            lambda x: x.rolling(window).std()
        )
    )

# Moving averages
for window in [5, 10, 20, 50]:
    daily_data[f"Close_MA_{window}"] = (
        daily_data.groupby("Ticker")["Close"]
        .transform(
            lambda x: x.rolling(window).mean()
        )
    )

# Moving average ratios
for window in [5, 10, 20, 50]:
    daily_data[f"Close_MA_Ratio_{window}"] = (
        (
            daily_data["Close"] /
            daily_data[f"Close_MA_{window}"] - 1
        ) * 100
    )

# Momentum
for period in [5, 10, 20]:
    daily_data[f"Close_Momentum_{period}D"] = (
        daily_data.groupby("Ticker")["Close"]
        .transform(
            lambda x: (x / x.shift(period) - 1) * 100
        )
    )

In [31]:
# Columns to inspect
close_features = [
    "Close_Daily_Return",

    "Close_Return_Lag_1",
    "Close_Return_Lag_5",
    "Close_Return_Lag_10",
    "Close_Return_Lag_20",

    "Close_Volatility_5D",
    "Close_Volatility_10D",
    "Close_Volatility_20D",

    "Close_MA_5",
    "Close_MA_10",
    "Close_MA_20",
    "Close_MA_50",

    "Close_MA_Ratio_5",
    "Close_MA_Ratio_10",
    "Close_MA_Ratio_20",
    "Close_MA_Ratio_50",

    "Close_Momentum_5D",
    "Close_Momentum_10D",
    "Close_Momentum_20D"
]

# 1. Dataset information
print("=" * 70)
print("CLOSE-BASED FEATURES SUMMARY")
print("=" * 70)

print("Dataset Shape:", daily_data.shape)

# 2. Missing values
print("\nMissing Values:")
print(
    daily_data[close_features]
    .isnull()
    .sum()
)

# 3. Descriptive statistics
print("\nDescriptive Statistics:")
print(
    daily_data[close_features]
    .describe()
    .T
    .round(6)
)

# 4. Compare selected Close and Adj Close features
comparison_columns = [
    "Daily_Return",
    "Close_Daily_Return",
    "MA_Ratio_20",
    "Close_MA_Ratio_20",
    "Momentum_20D",
    "Close_Momentum_20D"
]

print("\nClose vs Adj Close Comparison:")
print(
    daily_data[comparison_columns]
    .describe()
    .T
    .round(6)
)

# 5. Sample records
print("\nSample Records:")
print(
    daily_data[
        ["Date", "Ticker"] + close_features
    ].tail(10)
)

CLOSE-BASED FEATURES SUMMARY
Dataset Shape: (63574, 91)

Missing Values:
Close_Daily_Return       10
Close_Return_Lag_1       20
Close_Return_Lag_5       60
Close_Return_Lag_10     110
Close_Return_Lag_20     210
Close_Volatility_5D      50
Close_Volatility_10D    100
Close_Volatility_20D    200
Close_MA_5               40
Close_MA_10              90
Close_MA_20             190
Close_MA_50             490
Close_MA_Ratio_5         40
Close_MA_Ratio_10        90
Close_MA_Ratio_20       190
Close_MA_Ratio_50       490
Close_Momentum_5D        50
Close_Momentum_10D      100
Close_Momentum_20D      200
dtype: int64

Descriptive Statistics:
                        count       mean        std        min        25%  \
Close_Daily_Return    63564.0   0.062741   1.948448 -29.180437  -0.783749   
Close_Return_Lag_1    63554.0   0.062708   1.948492 -29.180437  -0.783914   
Close_Return_Lag_5    63514.0   0.063028   1.947114 -29.180437  -0.783162   
Close_Return_Lag_10   63464.0   0.063086   1.9473

#### Calculating RSI and MACD using the Close

In [32]:
# Calculate price changes
delta = daily_data.groupby("Ticker")["Close"].diff()

# Separate gains and losses
gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)

# Calculate exponential moving averages
avg_gain = gain.groupby(daily_data["Ticker"]).transform(
    lambda x: x.ewm(
        alpha=1/14,
        adjust=False,
        min_periods=14
    ).mean()
)

avg_loss = loss.groupby(daily_data["Ticker"]).transform(
    lambda x: x.ewm(
        alpha=1/14,
        adjust=False,
        min_periods=14
    ).mean()
)

# Calculate RSI
rs = avg_gain / avg_loss

daily_data["Close_RSI_14"] = (
    100 - (100 / (1 + rs))
)

# Handle zero-loss cases
daily_data.loc[
    (avg_loss == 0) & (avg_gain > 0),
    "Close_RSI_14"
] = 100

daily_data.loc[
    (avg_loss == 0) & (avg_gain == 0),
    "Close_RSI_14"
] = 50

In [33]:
# Calculate EMA 12
ema_12 = daily_data.groupby("Ticker")["Close"].transform(
    lambda x: x.ewm(
        span=12,
        adjust=False,
        min_periods=12
    ).mean()
)

# Calculate EMA 26
ema_26 = daily_data.groupby("Ticker")["Close"].transform(
    lambda x: x.ewm(
        span=26,
        adjust=False,
        min_periods=26
    ).mean()
)

# MACD
daily_data["Close_MACD"] = ema_12 - ema_26

# Signal Line
daily_data["Close_MACD_Signal"] = (
    daily_data.groupby("Ticker")["Close_MACD"].transform(
        lambda x: x.ewm(
            span=9,
            adjust=False,
            min_periods=9
        ).mean()
    )
)

# MACD Histogram
daily_data["Close_MACD_Histogram"] = (
    daily_data["Close_MACD"] -
    daily_data["Close_MACD_Signal"]
)

In [34]:
close_oscillators = [
    "Close_RSI_14",
    "Close_MACD",
    "Close_MACD_Signal",
    "Close_MACD_Histogram"
]

print("=" * 65)
print("CLOSE-BASED RSI AND MACD AUDIT")
print("=" * 65)

print("\nMissing Values:")
print(daily_data[close_oscillators].isnull().sum())

print("\nDescriptive Statistics:")
print(
    daily_data[close_oscillators]
    .describe()
    .T
    .round(6)
)

print("\nComparison with Previous Indicators:")

comparison = [
    "RSI_14",
    "Close_RSI_14",
    "MACD",
    "Close_MACD",
    "MACD_Signal",
    "Close_MACD_Signal",
    "MACD_Histogram",
    "Close_MACD_Histogram"
]

print(
    daily_data[comparison]
    .describe()
    .T
    .round(6)
)

CLOSE-BASED RSI AND MACD AUDIT

Missing Values:
Close_RSI_14            140
Close_MACD              250
Close_MACD_Signal       330
Close_MACD_Histogram    330
dtype: int64

Descriptive Statistics:
                        count       mean        std        min        25%  \
Close_RSI_14          63434.0  52.759536  11.868737   7.305392  44.478013   
Close_MACD            63324.0   0.147546   1.207408 -16.912446  -0.212940   
Close_MACD_Signal     63244.0   0.147123   1.136097 -15.094499  -0.198684   
Close_MACD_Histogram  63244.0   0.000647   0.366462  -5.673820  -0.100522   

                            50%        75%        max  
Close_RSI_14          52.914344  61.041943  90.940117  
Close_MACD             0.088750   0.469877  14.296355  
Close_MACD_Signal      0.086618   0.448007  11.995421  
Close_MACD_Histogram   0.000634   0.104727   4.865464  

Comparison with Previous Indicators:
                        count       mean        std        min        25%  \
RSI_14               

In [35]:
# Normalize MACD indicators relative to closing price

daily_data["MACD_Pct"] = (
    daily_data["Close_MACD"] /
    daily_data["Close"] * 100
)

daily_data["MACD_Signal_Pct"] = (
    daily_data["Close_MACD_Signal"] /
    daily_data["Close"] * 100
)

daily_data["MACD_Histogram_Pct"] = (
    daily_data["Close_MACD_Histogram"] /
    daily_data["Close"] * 100
)

# Print statistics
normalized_macd = [
    "MACD_Pct",
    "MACD_Signal_Pct",
    "MACD_Histogram_Pct"
]

print("Normalized MACD Statistics:")
display(
    daily_data[normalized_macd]
    .describe()
    .T
    .round(6)
)

print("\nMissing Values:")
print(daily_data[normalized_macd].isnull().sum())

Normalized MACD Statistics:


,count,mean,std,min,25%,50%,75%,max
MACD_Pct,63324.0,0.193780,2.176642,-60.830244,-0.684855,0.387187,1.291356,9.487040
MACD_Signal_Pct,63244.0,0.196399,2.027982,-47.044711,-0.630943,0.383968,1.234388,8.886252
MACD_Histogram_Pct,63244.0,-0.001734,0.653764,-17.063561,-0.305063,0.002874,0.315285,7.590817



Missing Values:
MACD_Pct              250
MACD_Signal_Pct       330
MACD_Histogram_Pct    330
dtype: int64


In [36]:
# Old Adj Close-based technical features
old_features = [
    "Daily_Return",

    "Return_Lag_1",
    "Return_Lag_5",
    "Return_Lag_10",
    "Return_Lag_20",

    "Volatility_5D",
    "Volatility_10D",
    "Volatility_20D",

    "MA_5",
    "MA_10",
    "MA_20",
    "MA_50",

    "MA_Ratio_5",
    "MA_Ratio_10",
    "MA_Ratio_20",
    "MA_Ratio_50",

    "Momentum_5D",
    "Momentum_10D",
    "Momentum_20D",

    "RSI_14",

    "MACD",
    "MACD_Signal",
    "MACD_Histogram"
]

# Unnecessary raw Close-based indicators
unnecessary_close_features = [
    "Close_MA_5",
    "Close_MA_10",
    "Close_MA_20",
    "Close_MA_50",

    "Close_MACD",
    "Close_MACD_Signal",
    "Close_MACD_Histogram"
]

columns_to_drop = old_features + unnecessary_close_features

daily_data = daily_data.drop(
    columns=columns_to_drop,
    errors="ignore"
)

print("Dataset Shape:", daily_data.shape)

print("\nRemaining Old Features:")
print([
    col for col in old_features
    if col in daily_data.columns
])

Dataset Shape: (63574, 68)

Remaining Old Features:
[]


In [37]:
technical_features = [
    "Close_Daily_Return",

    "Close_Return_Lag_1",
    "Close_Return_Lag_5",
    "Close_Return_Lag_10",
    "Close_Return_Lag_20",

    "Close_Volatility_5D",
    "Close_Volatility_10D",
    "Close_Volatility_20D",

    "Close_MA_Ratio_5",
    "Close_MA_Ratio_10",
    "Close_MA_Ratio_20",
    "Close_MA_Ratio_50",

    "Close_Momentum_5D",
    "Close_Momentum_10D",
    "Close_Momentum_20D",

    "Close_RSI_14",

    "MACD_Pct",
    "MACD_Signal_Pct",
    "MACD_Histogram_Pct"
]

print("Technical Features:", len(technical_features))

print("\nMissing Values:")
print(daily_data[technical_features].isnull().sum())

print("\nDescriptive Statistics:")
display(
    daily_data[technical_features]
    .describe()
    .T
    .round(6)
)

Technical Features: 19

Missing Values:
Close_Daily_Return       10
Close_Return_Lag_1       20
Close_Return_Lag_5       60
Close_Return_Lag_10     110
Close_Return_Lag_20     210
Close_Volatility_5D      50
Close_Volatility_10D    100
Close_Volatility_20D    200
Close_MA_Ratio_5         40
Close_MA_Ratio_10        90
Close_MA_Ratio_20       190
Close_MA_Ratio_50       490
Close_Momentum_5D        50
Close_Momentum_10D      100
Close_Momentum_20D      200
Close_RSI_14            140
MACD_Pct                250
MACD_Signal_Pct         330
MACD_Histogram_Pct      330
dtype: int64

Descriptive Statistics:


,count,mean,std,min,25%,50%,75%,max
Close_Daily_Return,63564.0,0.062741,1.948448,-29.180437,-0.783749,0.067043,0.895296,32.462690
Close_Return_Lag_1,63554.0,0.062708,1.948492,-29.180437,-0.783914,0.067066,0.895326,32.462690
Close_Return_Lag_5,63514.0,0.063028,1.947114,-29.180437,-0.783162,0.067225,0.895241,32.462690
Close_Return_Lag_10,63464.0,0.063086,1.947308,-29.180437,-0.783332,0.066952,0.895296,32.462690
Close_Return_Lag_20,63364.0,0.062588,1.947574,-29.180437,-0.783749,0.066664,0.894294,32.462690
Close_Volatility_5D,63524.0,1.531759,1.222011,0.042577,0.766555,1.223346,1.913271,16.576033
Close_Volatility_10D,63474.0,1.598589,1.127496,0.164522,0.884926,1.323528,1.956484,14.525634
Close_Volatility_20D,63374.0,1.637724,1.064883,0.237631,0.955714,1.384229,1.985775,13.870796
Close_MA_Ratio_5,63534.0,0.094204,2.095930,-28.822905,-0.854699,0.124909,1.057819,26.172876
Close_MA_Ratio_10,63484.0,0.218748,3.166656,-45.759363,-1.266924,0.278296,1.738341,40.081501


In [40]:
# Select MACD-related columns
macd_columns = [
    "MACD_Pct",
    "MACD_Signal_Pct",
    "MACD_Histogram_Pct"
]

print("=" * 75)
print("MACD EXTREME VALUES INVESTIGATION")
print("=" * 75)

for col in macd_columns:

    print(f"\n{'=' * 60}")
    print(f"Feature: {col}")
    print("=" * 60)

    # Lowest 15 values
    print("\nLowest 15 Values:")

    print(
        daily_data.nsmallest(15, col)[
            ["Date", "Ticker", "Close", col]
        ]
    )

    # Highest 15 values
    print("\nHighest 15 Values:")

    print(
        daily_data.nlargest(15, col)[
            ["Date", "Ticker", "Close", col]
        ]
    )

MACD EXTREME VALUES INVESTIGATION

Feature: MACD_Pct

Lowest 15 Values:
            Date  Ticker      Close   MACD_Pct
25310 2020-03-19  CNQ.TO   5.484819 -60.830244
25309 2020-03-18  CNQ.TO   5.386876 -59.278639
25311 2020-03-20  CNQ.TO   5.950049 -56.696311
25312 2020-03-23  CNQ.TO   5.959843 -56.431766
25313 2020-03-24  CNQ.TO   6.704212 -48.589381
25308 2020-03-17  CNQ.TO   6.346719 -46.545631
25314 2020-03-25  CNQ.TO   6.988247 -44.573889
25316 2020-03-27  CNQ.TO   6.523017 -43.602460
25315 2020-03-26  CNQ.TO   7.174339 -41.158715
55166 2020-03-23   SU.TO  15.410000 -40.852959
55164 2020-03-19   SU.TO  15.630000 -38.961984
55165 2020-03-20   SU.TO  16.070000 -38.676283
55163 2020-03-18   SU.TO  15.070000 -38.461647
25307 2020-03-16  CNQ.TO   7.071499 -38.232953
55167 2020-03-24   SU.TO  17.420000 -35.167401

Highest 15 Values:
            Date   Ticker       Close  MACD_Pct
48447 2020-04-28  SHOP.TO   88.642998  9.487040
48448 2020-04-29  SHOP.TO   88.842003  9.444553
48446 2020-0

In [41]:
macd_by_ticker = (
    daily_data.groupby("Ticker")[macd_columns]
    .agg(["mean", "std", "min", "median", "max"])
    .round(4)
)

display(macd_by_ticker)

MACD_Pct                                  MACD_Signal_Pct          \
            mean     std      min  median     max            mean     std   
Ticker                                                                      
ABX.TO   -0.0965  2.6337 -17.2737  0.0688  8.3469         -0.0917  2.4584   
BMO.TO    0.1779  1.5850 -18.6759  0.3000  5.4015          0.1795  1.4757   
BN.TO     0.3059  1.7493 -15.3476  0.5381  5.3011          0.3102  1.6261   
CNQ.TO    0.1624  3.1766 -60.8302  0.4755  6.7958          0.1697  2.9052   
CNR.TO    0.2970  1.3271  -5.7608  0.3944  4.4414          0.3006  1.2201   
ENB.TO    0.1951  1.2813 -13.7921  0.3742  3.8440          0.1981  1.1853   
RY.TO     0.2572  1.3236  -9.8364  0.3680  5.0903          0.2577  1.2429   
SHOP.TO   0.5857  4.0125 -21.2957  1.2683  9.4870          0.5782  3.8126   
SU.TO     0.0983  2.8060 -40.8530  0.4130  7.0397          0.1002  2.6313   
TD.TO     0.1802  1.4167 -11.0036  0.3483  4.7331          0.1814  1.3233   

                                 MACD_Histogram_Pct                           \
             min  median     max               mean     std      min  median   
Ticker                                                                         
ABX.TO  -14.2395  0.0691  7.8956            -0.0059  0.8165  -5.4799  0.0123   
BMO.TO  -15.9020  0.3017  5.0798            -0.0006  0.4825  -5.1556 -0.0007   
BN.TO   -10.5939  0.5585  4.6052            -0.0003  0.5572  -5.7329 -0.0057   
CNQ.TO  -47.0447  0.4817  5.9556            -0.0044  0.9228 -17.0636 -0.0060   
CNR.TO   -5.0789  0.3693  4.0406            -0.0019  0.4575  -2.2310  0.0029   
ENB.TO  -10.9920  0.3678  3.4658            -0.0015  0.4245  -4.4047  0.0069   
RY.TO    -7.4922  0.3841  4.6548             0.0007  0.3962  -3.0150 -0.0056   
SHOP.TO -18.9758  1.2390  8.8863            -0.0044  1.1792  -4.5029  0.0697   
SU.TO   -33.1202  0.4372  6.2084            -0.0006  0.7997 -11.6233  0.0094   
TD.TO    -9.1136  0.3215  4.2000             0.0000  0.4374  -3.6573  0.0021   

                 
            max  
Ticker           
ABX.TO   3.4882  
BMO.TO   2.9874  
BN.TO    3.0902  
CNQ.TO   7.5908  
CNR.TO   1.9869  
ENB.TO   1.6768  
RY.TO    2.4027  
SHOP.TO  4.2734  
SU.TO    6.0388  
TD.TO    2.5728

In [44]:
print("Dataset Shape:", daily_data.shape)

print("\nMACD Features:")
print([
    col for col in daily_data.columns
    if "MACD" in col
])

print("\nTarget:")
print("Future_Return_5D" in daily_data.columns)

Dataset Shape: (63574, 68)

MACD Features:
['MACD_Pct', 'MACD_Signal_Pct', 'MACD_Histogram_Pct']

Target:
True


In [45]:
column_audit = pd.DataFrame({
    "Column": daily_data.columns,
    "Data_Type": daily_data.dtypes.astype(str).values,
    "Missing_Values": daily_data.isnull().sum().values,
    "Missing_Percentage": (
        daily_data.isnull().mean().values * 100
    ).round(2)
})

print(column_audit.to_string(index=False))

                         Column      Data_Type  Missing_Values  Missing_Percentage
                           Date datetime64[ns]               0                0.00
                         Ticker         object               0                0.00
                           Open        float64               0                0.00
                           High        float64               0                0.00
                            Low        float64               0                0.00
                          Close        float64               0                0.00
                      Adj Close        float64               0                0.00
                         Volume          int64               0                0.00
                      Dividends        float64               0                0.00
                   Stock Splits        float64               0                0.00
         Bank_Availability_Date datetime64[ns]            2259                3.55
    

In [46]:
print("\nAll Columns:")
for i, col in enumerate(daily_data.columns, 1):
    print(f"{i:02d}. {col}")


All Columns:
01. Date
02. Ticker
03. Open
04. High
05. Low
06. Close
07. Adj Close
08. Volume
09. Dividends
10. Stock Splits
11. Bank_Availability_Date
12. Policy_Rate
13. Bond_Yield_10Y
14. USD_CAD
15. Macro_Reference_Date
16. CPI
17. Inflation_YoY
18. GDP_Real_Millions_CAD
19. GDP_MoM
20. GDP_YoY
21. Employment
22. Labour_Force
23. Participation_Rate
24. Unemployment_Rate
25. BCPI_Total
26. BCPI_NonEnergy
27. BCPI_Energy
28. BCPI_Metals
29. BCPI_Forestry
30. BCPI_Agriculture
31. BCPI_Fish
32. Macro_Availability_Date
33. DayOfWeek
34. Month
35. Quarter
36. DayOfWeek_Sin
37. DayOfWeek_Cos
38. Month_Sin
39. Month_Cos
40. Macro_Feature_Availability_Date
41. BCPI_Change_12M
42. BCPI_Energy_Change_12M
43. BCPI_Metals_Change_12M
44. Unemployment_Change_12M
45. Employment_Growth_12M
46. Yield_Spread
47. Policy_Rate_Change_21D
48. USD_CAD_Change_21D
49. Future_Return_5D
50. Close_Daily_Return
51. Close_Return_Lag_1
52. Close_Return_Lag_5
53. Close_Return_Lag_10
54. Close_Return_Lag_20
55. Cl

In [48]:
DATA_DIR = "data/processed"
output_path = DATA_DIR + "canada_integrated_daily_macro_features_final.csv"

daily_data.to_csv(output_path, index=False)

print("Saved successfully:", output_path)


Saved successfully: data/processedcanada_integrated_daily_macro_features_demo2.csv


# Temporal Leakage Audit

## 1. Objective

The Temporal Leakage Audit was conducted to verify that financial and macroeconomic features were not assigned to stock records using information from future dates. This is a critical step in ensuring that the machine learning model uses only information that would have been available at the time of prediction.

## 2. Audit Methodology

The audit examined the temporal alignment of three groups of data:

- **Bank of Canada Data:** Daily financial indicators, including the policy rate, 10-year bond yield, and USD/CAD exchange rate.
- **Monthly Macroeconomic Data:** CPI, GDP, labour market indicators, and the Bank of Canada Commodity Price Index (BCPI).
- **Engineered Macroeconomic Features:** Additional features derived from monthly economic indicators, including 12-month changes and growth rates.

The integration process used backward merge_asof operations to associate each stock observation with the most recent eligible economic observation.

For daily Bank of Canada indicators, a maximum matching tolerance of seven days was applied.

For monthly macroeconomic indicators and their engineered features, a temporary three-month availability delay was applied before integration. This was intended to reduce the risk of using unreleased economic information.

## 3. Audit Results

The following checks were performed:

| Audit Check                                | Violations |
| ------------------------------------------ | ---------- |
| Bank of Canada Availability Violations     | 0          |
| Monthly Macro Availability Violations      | 0          |
| Macro Feature Availability Violations      | 0          |
| Macro Reference Date Violations            | 0          |
| Macro Feature Availability Date Mismatches | 0          |

All implemented temporal checks passed without detecting future-date violations.

### Bank Data Age Analysis

The age of matched Bank of Canada observations was also examined.

| Statistic          | Data Age      |
| ------------------ | ------------- |
| Count              | 61,315        |
| Mean               | 0.002675 days |
| Standard Deviation | 0.078679 days |
| Minimum            | 0 days        |
| 25th Percentile    | 0 days        |
| Median             | 0 days        |
| 75th Percentile    | 0 days        |
| Maximum            | 3 days        |

No matched Bank of Canada observation was older than seven days.

A total of 2,259 stock records had no matching Bank availability date, indicating that a daily Bank of Canada observation could not be matched within the specified tolerance.

## 4. Interpretation

The results indicate that the implemented temporal integration logic successfully prevented the use of observations dated after the corresponding stock records.

The backward matching strategy ensured that financial indicators were matched using previously available observations rather than future observations.

Furthermore, the Bank of Canada data-age analysis confirmed that all successfully matched observations were within the defined seven-day tolerance.

However, an important limitation remains regarding monthly macroeconomic data.

The three-month availability delay is a conservative methodological assumption, not a verified record of the actual publication date of each economic indicator. The reference date of a monthly observation does not necessarily represent the date on which that information became publicly available.

Therefore, the absence of temporal violations confirms the correctness of the implemented date-matching rules, but does not independently establish that every monthly observation was historically available on its assigned availability date.

## 5. Conclusion

The Temporal Leakage Audit passed all implemented validation checks:

- No Bank of Canada availability violations were detected.
- No monthly macroeconomic availability violations were detected.
- No engineered macroeconomic feature availability violations were detected.
- No macro reference-date violations were detected.
- All matched daily Bank of Canada observations were within seven days of their corresponding stock dates.

The integration process is therefore temporally consistent under the assumptions currently implemented.

Before final model evaluation and historical backtesting, the assumed three-month macroeconomic delay should be verified against official publication dates or replaced with defensible series-specific publication lags.

This additional verification is necessary to strengthen the project's protection against look-ahead bias.

# Temporal Leakage Audit – Summary

A temporal leakage audit was performed to verify the chronological alignment of daily financial data, monthly macroeconomic indicators, and engineered economic features.

Backward merge_asof operations were used to ensure that stock observations were matched with previously dated economic information. Daily Bank of Canada data was integrated using a seven-day tolerance, while monthly macroeconomic data used a temporary three-month availability delay.

All implemented checks passed:

- Bank availability violations: 0
- Macro availability violations: 0
- Macro feature availability violations: 0
- Macro reference date violations: 0
- Macro availability date mismatches: 0

The matched Bank of Canada observations had a maximum age of three days, with no records exceeding the seven-day tolerance.

Overall, the audit confirmed that the implemented integration rules did not introduce future-date matching. However, the three-month macroeconomic delay remains an assumption that requires verification against official publication dates before final model evaluation and backtesting.

In [4]:
DATA_DIR = "data/processed"
daily_data = pd.read_csv(
    DATA_DIR + "canada_integrated_daily_macro_features_final.csv",
    parse_dates=[
        "Date",
        "Bank_Availability_Date",
        "Macro_Reference_Date",
        "Macro_Availability_Date"
    ]
)

daily_data = daily_data.sort_values(
    ["Ticker", "Date"]
).reset_index(drop=True)

print("Shape:", daily_data.shape)
print("Tickers:", daily_data["Ticker"].nunique())
print("Date Range:", daily_data["Date"].min(), "to", daily_data["Date"].max())

daily_data.head()

Shape: (63574, 68)
Tickers: 10
Date Range: 2000-01-04 00:00:00 to 2026-11-06 00:00:00


,Date,Ticker,Open,High,Low,Close,Adj Close,Volume,Dividends,Stock Splits,...,Close_MA_Ratio_10,Close_MA_Ratio_20,Close_MA_Ratio_50,Close_Momentum_5D,Close_Momentum_10D,Close_Momentum_20D,Close_RSI_14,MACD_Pct,MACD_Signal_Pct,MACD_Histogram_Pct
0,2000-01-04,ABX.TO,25.150000,25.65,24.950001,25.450001,17.804482,926500,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2000-01-05,ABX.TO,25.400000,26.00,25.350000,25.650000,17.944401,1452200,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2000-01-06,ABX.TO,25.799999,26.00,25.500000,25.799999,18.049345,600200,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2000-01-07,ABX.TO,25.850000,26.00,25.100000,25.250000,17.664574,1020800,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2000-01-10,ABX.TO,25.350000,25.35,25.150000,25.250000,17.664574,1281100,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
technical_features = [
    "Close_Daily_Return",
    "Close_Return_Lag_1",
    "Close_Return_Lag_5",
    "Close_Return_Lag_10",
    "Close_Return_Lag_20",
    "Close_Volatility_5D",
    "Close_Volatility_10D",
    "Close_Volatility_20D",
    "Close_MA_Ratio_5",
    "Close_MA_Ratio_10",
    "Close_MA_Ratio_20",
    "Close_MA_Ratio_50",
    "Close_Momentum_5D",
    "Close_Momentum_10D",
    "Close_Momentum_20D",
    "Close_RSI_14",
    "MACD_Pct",
    "MACD_Signal_Pct",
    "MACD_Histogram_Pct"
]

In [6]:
economic_features = [
    "Policy_Rate",
    "Bond_Yield_10Y",
    "USD_CAD",
    "CPI",
    "Inflation_YoY",
    "GDP_Real_Millions_CAD",
    "GDP_MoM",
    "GDP_YoY",
    "Employment",
    "Labour_Force",
    "Participation_Rate",
    "Unemployment_Rate",
    "BCPI_Total",
    "BCPI_NonEnergy",
    "BCPI_Energy",
    "BCPI_Metals",
    "BCPI_Forestry",
    "BCPI_Agriculture",
    "BCPI_Fish",
    "BCPI_Change_12M",
    "BCPI_Energy_Change_12M",
    "BCPI_Metals_Change_12M",
    "Unemployment_Change_12M",
    "Employment_Growth_12M",
    "Yield_Spread",
    "Policy_Rate_Change_21D",
    "USD_CAD_Change_21D"
]

In [7]:
calendar_features = [
    "DayOfWeek_Sin",
    "DayOfWeek_Cos",
    "Month_Sin",
    "Month_Cos"
]

In [8]:
# Combine feature groups
feature_columns = (
    technical_features +
    economic_features +
    calendar_features
)

# Create independent copy
final_data = daily_data.copy(deep=True)

# Separate features and target
X = final_data[feature_columns].copy(deep=True)
y = final_data["Future_Return_5D"].copy(deep=True)

print("Final Dataset Shape:", final_data.shape)
print("Number of Candidate Features:", len(feature_columns))
print("X Shape:", X.shape)
print("y Shape:", y.shape)

print("\nTarget Missing Values:", y.isna().sum())

print("\nFeature Missing Values:")
print(X.isnull().sum()[X.isnull().sum() > 0])

Final Dataset Shape: (63574, 68)
Number of Candidate Features: 50
X Shape: (63574, 50)
y Shape: (63574,)

Target Missing Values: 50

Feature Missing Values:
Close_Daily_Return           10
Close_Return_Lag_1           20
Close_Return_Lag_5           60
Close_Return_Lag_10         110
Close_Return_Lag_20         210
Close_Volatility_5D          50
Close_Volatility_10D        100
Close_Volatility_20D        200
Close_MA_Ratio_5             40
Close_MA_Ratio_10            90
Close_MA_Ratio_20           190
Close_MA_Ratio_50           490
Close_Momentum_5D            50
Close_Momentum_10D          100
Close_Momentum_20D          200
Close_RSI_14                140
MACD_Pct                    250
MACD_Signal_Pct             330
MACD_Histogram_Pct          330
Policy_Rate               21090
Bond_Yield_10Y             2473
USD_CAD                   38984
Yield_Spread              21294
Policy_Rate_Change_21D    21320
USD_CAD_Change_21D        39324
dtype: int64


In [10]:
DATA_DIR = "data/processed/"
output_path = DATA_DIR + "canada_integrated_daily_macro_features_base50.csv"

final_data.to_csv(output_path, index=False)

print("Saved successfully:", output_path)

Saved successfully: data/processed/canada_integrated_daily_macro_features_base50.csv
